# Mizo PoS Tagging - Data Preparation

## Cell 1: Data Loading and Exploration

In [1]:
"""
Cell 1: Data Loading and Exploration
=====================================
Upload your files to the same directory or update paths below.
Files expected:
  - small.mz, small.en (verified parallel corpus)
  - large.mz, large.en (unverified parallel corpus)
  - mizo_words_ud.txt (word-level PoS lexicon)
  - mizo_phrases_ud.txt (phrase-level PoS lexicon)
"""

import os

# ============================================================
# UPDATE THESE PATHS to where your files are located
# ============================================================
DATA_DIR = "data/"  # Change if needed

small_mz_path = os.path.join(DATA_DIR, "small.mz")
small_en_path = os.path.join(DATA_DIR, "small.en")
large_mz_path = os.path.join(DATA_DIR, "large.mz")
large_en_path = os.path.join(DATA_DIR, "large.en")
words_ud_path = os.path.join(DATA_DIR, "mizo_words_ud.txt")
phrases_ud_path = os.path.join(DATA_DIR, "mizo_phrases_ud.txt")

# ============================================================
# Helper: read lines from file
# ============================================================
def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

# ============================================================
# 1. Check file existence
# ============================================================
print("=" * 60)
print("FILE EXISTENCE CHECK")
print("=" * 60)
all_files = {
    "small.mz": small_mz_path,
    "small.en": small_en_path,
    "large.mz": large_mz_path,
    "large.en": large_en_path,
    "mizo_words_ud.txt": words_ud_path,
    "mizo_phrases_ud.txt": phrases_ud_path,
}

missing = []
for name, path in all_files.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) / (1024 * 1024) if exists else 0
    status = f"✓ ({size:.2f} MB)" if exists else "✗ MISSING"
    print(f"  {name:25s} {status}")
    if not exists:
        missing.append(name)

if missing:
    print(f"\n⚠️  Missing files: {missing}")
    print("Please upload them and re-run this cell.")
else:
    print("\nAll files found!")

# ============================================================
# 2. Load and verify parallel corpora
# ============================================================
print("\n" + "=" * 60)
print("PARALLEL CORPORA STATISTICS")
print("=" * 60)

if not missing:
    small_mz = read_lines(small_mz_path)
    small_en = read_lines(small_en_path)
    large_mz = read_lines(large_mz_path)
    large_en = read_lines(large_en_path)

    print(f"\n  Small corpus:")
    print(f"    Mizo sentences:   {len(small_mz):>10,}")
    print(f"    English sentences:{len(small_en):>10,}")
    print(f"    Aligned:          {'✓ YES' if len(small_mz) == len(small_en) else '✗ MISMATCH!'}")

    print(f"\n  Large corpus:")
    print(f"    Mizo sentences:   {len(large_mz):>10,}")
    print(f"    English sentences:{len(large_en):>10,}")
    print(f"    Aligned:          {'✓ YES' if len(large_mz) == len(large_en) else '✗ MISMATCH!'}")

    # Sample sentences
    print("\n  --- Small Corpus Samples (first 3 pairs) ---")
    for i in range(min(3, len(small_mz))):
        print(f"    MZ: {small_mz[i][:100]}")
        print(f"    EN: {small_en[i][:100]}")
        print()

    print("  --- Large Corpus Samples (first 3 pairs) ---")
    for i in range(min(3, len(large_mz))):
        print(f"    MZ: {large_mz[i][:100]}")
        print(f"    EN: {large_en[i][:100]}")
        print()

    # Sentence length statistics
    import statistics
    small_mz_lens = [len(s.split()) for s in small_mz]
    small_en_lens = [len(s.split()) for s in small_en]
    large_mz_lens = [len(s.split()) for s in large_mz]
    large_en_lens = [len(s.split()) for s in large_en]

    print("  Sentence length (words) statistics:")
    print(f"    {'Corpus':<12} {'Min':>6} {'Max':>6} {'Mean':>8} {'Median':>8}")
    print(f"    {'small.mz':<12} {min(small_mz_lens):>6} {max(small_mz_lens):>6} {statistics.mean(small_mz_lens):>8.1f} {statistics.median(small_mz_lens):>8.1f}")
    print(f"    {'small.en':<12} {min(small_en_lens):>6} {max(small_en_lens):>6} {statistics.mean(small_en_lens):>8.1f} {statistics.median(small_en_lens):>8.1f}")
    print(f"    {'large.mz':<12} {min(large_mz_lens):>6} {max(large_mz_lens):>6} {statistics.mean(large_mz_lens):>8.1f} {statistics.median(large_mz_lens):>8.1f}")
    print(f"    {'large.en':<12} {min(large_en_lens):>6} {max(large_en_lens):>6} {statistics.mean(large_en_lens):>8.1f} {statistics.median(large_en_lens):>8.1f}")

    # Empty line check
    small_mz_empty = sum(1 for s in small_mz if len(s.strip()) == 0)
    small_en_empty = sum(1 for s in small_en if len(s.strip()) == 0)
    large_mz_empty = sum(1 for s in large_mz if len(s.strip()) == 0)
    large_en_empty = sum(1 for s in large_en if len(s.strip()) == 0)
    print(f"\n  Empty lines: small.mz={small_mz_empty}, small.en={small_en_empty}, "
          f"large.mz={large_mz_empty}, large.en={large_en_empty}")

# ============================================================
# 3. Load and analyze PoS lexicons
# ============================================================
print("\n" + "=" * 60)
print("PoS LEXICON STATISTICS")
print("=" * 60)

if not missing:
    # Load word-level lexicon
    words_data = []
    words_parse_errors = 0
    with open(words_ud_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) == 2:
                words_data.append((parts[0], parts[1]))
            else:
                words_parse_errors += 1
                if words_parse_errors <= 3:
                    print(f"    ⚠️  Word lexicon parse issue line {i+1}: {repr(line[:80])}")

    # Load phrase-level lexicon
    phrases_data = []
    phrases_parse_errors = 0
    with open(phrases_ud_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) == 2:
                phrases_data.append((parts[0], parts[1]))
            else:
                phrases_parse_errors += 1
                if phrases_parse_errors <= 3:
                    print(f"    ⚠️  Phrase lexicon parse issue line {i+1}: {repr(line[:80])}")

    print(f"\n  mizo_words_ud.txt:")
    print(f"    Valid entries:    {len(words_data):>10,}")
    print(f"    Parse errors:    {words_parse_errors:>10,}")
    print(f"    Unique words:    {len(set(w for w, t in words_data)):>10,}")

    # Tag distribution for words
    from collections import Counter
    word_tag_counts = Counter(t for w, t in words_data)
    print(f"    Unique UD tags:  {len(word_tag_counts):>10}")
    print(f"    Tag distribution:")
    for tag, count in word_tag_counts.most_common():
        print(f"      {tag:<12} {count:>8,} ({100*count/len(words_data):>5.1f}%)")

    print(f"\n  mizo_phrases_ud.txt:")
    print(f"    Valid entries:    {len(phrases_data):>10,}")
    print(f"    Parse errors:    {phrases_parse_errors:>10,}")

    phrase_tag_counts = Counter(t for w, t in phrases_data)
    print(f"    Unique UD tags:  {len(phrase_tag_counts):>10}")
    print(f"    Tag distribution:")
    for tag, count in phrase_tag_counts.most_common():
        print(f"      {tag:<12} {count:>8,} ({100*count/len(phrases_data):>5.1f}%)")

    # Show samples
    print(f"\n  --- Word Lexicon Samples (first 10) ---")
    for word, tag in words_data[:10]:
        print(f"    {word:<30} → {tag}")

    print(f"\n  --- Phrase Lexicon Samples (first 10) ---")
    for phrase, tag in phrases_data[:10]:
        print(f"    {phrase:<40} → {tag}")

print("\n" + "=" * 60)
print("Cell 1 Complete. Please report results before proceeding.")
print("=" * 60)

FILE EXISTENCE CHECK
  small.mz                  ✓ (0.76 MB)
  small.en                  ✓ (0.75 MB)
  large.mz                  ✓ (72.48 MB)
  large.en                  ✓ (72.49 MB)
  mizo_words_ud.txt         ✓ (0.72 MB)
  mizo_phrases_ud.txt       ✓ (0.45 MB)

All files found!

PARALLEL CORPORA STATISTICS

  Small corpus:
    Mizo sentences:       13,155
    English sentences:    13,155
    Aligned:          ✓ YES

  Large corpus:
    Mizo sentences:    1,357,838
    English sentences: 1,357,838
    Aligned:          ✓ YES

  --- Small Corpus Samples (first 3 pairs) ---
    MZ: ‘chumi chuan chu chu a ṭha a ti’ tih ringawt pawh hi kan buaipui hrep peih zêl a!
    EN: We are always ready to make a huge fuss over the mere fact that 'so-and-so likes such-and-such'!

    MZ: a hrufai a, isua thlalâk chu thih pawh hlau lo chuan a fâwp ta vawng vawng mai a!
    EN: He wiped it clean, and fearing not even death, he kissed the picture of Jesus tenderly!

    MZ: hritlang natna thlen theitu h

## Cell 2: Parse and Clean PoS Lexicons

In [3]:
"""
Cell 2: Parse and Clean PoS Lexicons
Cell 2 will: properly parse both lexicon files by detecting the actual delimiter and format, then export clean versions.
======================================
The lexicon files use comma (,) as delimiter, not tab.
Format: "word_or_phrase, [optional_metadata,] UD_TAG"
The UD tag is always the LAST comma-separated field.
"""

import os
from collections import Counter

# ============================================================
# UPDATE THESE PATHS
# ============================================================
DATA_DIR = "data/"  # Change to your local path
OUTPUT_DIR = "data2/"  # Change to your local path

words_ud_path = os.path.join(DATA_DIR, "mizo_words_ud.txt")
phrases_ud_path = os.path.join(DATA_DIR, "mizo_phrases_ud.txt")

# Known UD tags (Universal Dependencies v2)
VALID_UD_TAGS = {
    "ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN",
    "NUM", "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM",
    "VERB", "X"
}

def parse_lexicon_file(filepath, file_label=""):
    """
    Parse lexicon file where:
    - Lines may be wrapped in quotes
    - Fields are comma-separated
    - The LAST field is the UD tag
    - The FIRST field(s) before the last are the word/phrase
    - There may be metadata between word and tag
    """
    entries = []
    errors = []
    
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            
            # Remove surrounding quotes if present
            if line.startswith('"') and line.endswith('"'):
                line = line[1:-1].strip()
            elif line.startswith('"'):
                line = line[1:].strip()
            elif line.endswith('"'):
                line = line[:-1].strip()
            
            # Split by comma
            parts = [p.strip() for p in line.split(",")]
            
            if len(parts) < 2:
                errors.append((line_num, line, "Too few fields"))
                continue
            
            # The UD tag is the last field (strip any remaining quotes)
            tag = parts[-1].strip().strip('"').strip()
            
            if tag not in VALID_UD_TAGS:
                # Sometimes the tag might have extra characters
                # Try to find a valid tag in the last few fields
                found_tag = None
                found_idx = None
                for i in range(len(parts) - 1, -1, -1):
                    candidate = parts[i].strip().strip('"').strip()
                    if candidate in VALID_UD_TAGS:
                        found_tag = candidate
                        found_idx = i
                        break
                
                if found_tag:
                    tag = found_tag
                    # Word is everything before the tag field
                    word = ", ".join(parts[:found_idx]).strip()
                else:
                    errors.append((line_num, line, f"No valid UD tag found (last field: '{tag}')"))
                    continue
            else:
                # Word/phrase is everything before the tag
                # But we need to separate the actual word from metadata
                # The word is the first field, metadata are middle fields
                word = parts[0].strip()
            
            if not word:
                errors.append((line_num, line, "Empty word"))
                continue
            
            # Extract metadata if present (middle fields between word and tag)
            metadata = ""
            tag_idx = len(parts) - 1
            # Find actual tag index
            for i in range(len(parts) - 1, 0, -1):
                if parts[i].strip().strip('"') == tag:
                    tag_idx = i
                    break
            
            if tag_idx > 1:
                metadata = ", ".join(parts[1:tag_idx]).strip()
            
            entries.append({
                "word": word,
                "tag": tag,
                "metadata": metadata,
                "line_num": line_num
            })
    
    return entries, errors

# ============================================================
# Parse both files
# ============================================================
print("=" * 60)
print("PARSING LEXICON FILES")
print("=" * 60)

word_entries, word_errors = parse_lexicon_file(words_ud_path, "words")
phrase_entries, phrase_errors = parse_lexicon_file(phrases_ud_path, "phrases")

print(f"\n  mizo_words_ud.txt:")
print(f"    Successfully parsed: {len(word_entries):>10,}")
print(f"    Parse errors:        {len(word_errors):>10,}")

print(f"\n  mizo_phrases_ud.txt:")
print(f"    Successfully parsed: {len(phrase_entries):>10,}")
print(f"    Parse errors:        {len(phrase_errors):>10,}")

# Show first few errors for debugging
if word_errors:
    print(f"\n  --- First 5 word lexicon errors ---")
    for line_num, line, reason in word_errors[:5]:
        print(f"    Line {line_num}: {reason}")
        print(f"      Raw: {repr(line[:100])}")

if phrase_errors:
    print(f"\n  --- First 5 phrase lexicon errors ---")
    for line_num, line, reason in phrase_errors[:5]:
        print(f"    Line {line_num}: {reason}")
        print(f"      Raw: {repr(line[:100])}")

# ============================================================
# Tag distribution analysis
# ============================================================
print("\n" + "=" * 60)
print("TAG DISTRIBUTIONS")
print("=" * 60)

word_tag_counts = Counter(e["tag"] for e in word_entries)
phrase_tag_counts = Counter(e["tag"] for e in phrase_entries)

print(f"\n  Word lexicon ({len(word_tag_counts)} tags):")
for tag, count in word_tag_counts.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/len(word_entries):>5.1f}%)")

print(f"\n  Phrase lexicon ({len(phrase_tag_counts)} tags):")
for tag, count in phrase_tag_counts.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/len(phrase_entries):>5.1f}%)")

# ============================================================
# Show samples from parsed data
# ============================================================
print("\n" + "=" * 60)
print("PARSED SAMPLES")
print("=" * 60)

print("\n  --- Word Lexicon (first 15) ---")
print(f"    {'Word':<25} {'Tag':<8} {'Metadata'}")
for e in word_entries[:15]:
    print(f"    {e['word']:<25} {e['tag']:<8} {e['metadata']}")

print(f"\n  --- Phrase Lexicon (first 15) ---")
print(f"    {'Phrase':<35} {'Tag':<8} {'Metadata'}")
for e in phrase_entries[:15]:
    print(f"    {e['phrase'] if 'phrase' in e else e['word']:<35} {e['tag']:<8} {e['metadata']}")

# ============================================================
# Check for duplicate words with different tags (ambiguous words)
# ============================================================
print("\n" + "=" * 60)
print("AMBIGUITY ANALYSIS")
print("=" * 60)

from collections import defaultdict
word_to_tags = defaultdict(set)
for e in word_entries:
    word_to_tags[e["word"].lower()].add(e["tag"])

ambiguous_words = {w: tags for w, tags in word_to_tags.items() if len(tags) > 1}
print(f"\n  Words with multiple tags: {len(ambiguous_words):,}")
print(f"  Total unique words:       {len(word_to_tags):,}")

if ambiguous_words:
    print(f"\n  --- Ambiguous words (first 20) ---")
    for i, (word, tags) in enumerate(sorted(ambiguous_words.items())[:20]):
        print(f"    {word:<25} → {', '.join(sorted(tags))}")

# ============================================================
# Export clean lexicons (tab-separated: word\ttag)
# ============================================================
print("\n" + "=" * 60)
print("EXPORTING CLEAN LEXICONS")
print("=" * 60)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Export word lexicon
clean_words_path = os.path.join(OUTPUT_DIR, "mizo_words_clean.tsv")
with open(clean_words_path, "w", encoding="utf-8") as f:
    f.write("word\ttag\n")  # header
    for e in word_entries:
        f.write(f"{e['word']}\t{e['tag']}\n")
print(f"  ✓ Exported: {clean_words_path} ({len(word_entries):,} entries)")

# Export phrase lexicon
clean_phrases_path = os.path.join(OUTPUT_DIR, "mizo_phrases_clean.tsv")
with open(clean_phrases_path, "w", encoding="utf-8") as f:
    f.write("phrase\ttag\n")  # header
    for e in phrase_entries:
        f.write(f"{e['word']}\t{e['tag']}\n")
print(f"  ✓ Exported: {clean_phrases_path} ({len(phrase_entries):,} entries)")

# Export combined lookup dictionary (word → most common tag)
# For ambiguous words, keep all tags
combined_path = os.path.join(OUTPUT_DIR, "mizo_lexicon_lookup.tsv")
with open(combined_path, "w", encoding="utf-8") as f:
    f.write("token\ttag\tsource\n")
    for e in word_entries:
        f.write(f"{e['word']}\t{e['tag']}\tword\n")
    for e in phrase_entries:
        f.write(f"{e['word']}\t{e['tag']}\tphrase\n")
print(f"  ✓ Exported: {combined_path} ({len(word_entries) + len(phrase_entries):,} entries)")

print("\n" + "=" * 60)
print("Cell 2 Complete. Please report results before proceeding.")
print("=" * 60)

PARSING LEXICON FILES

  mizo_words_ud.txt:
    Successfully parsed:     48,524
    Parse errors:                 0

  mizo_phrases_ud.txt:
    Successfully parsed:     25,630
    Parse errors:                 1

  --- First 5 phrase lexicon errors ---
    Line 13646: No valid UD tag found (last field: 'ud_tag')
      Raw: 'tokens, ud_tag'

TAG DISTRIBUTIONS

  Word lexicon (12 tags):
    NOUN         16,732 ( 34.5%)
    ADJ          15,453 ( 31.8%)
    VERB         13,817 ( 28.5%)
    ADV           1,539 (  3.2%)
    X               305 (  0.6%)
    PRON            244 (  0.5%)
    CCONJ           166 (  0.3%)
    NUM              87 (  0.2%)
    INTJ             73 (  0.2%)
    ADP              64 (  0.1%)
    PART             42 (  0.1%)
    PUNCT             2 (  0.0%)

  Phrase lexicon (12 tags):
    VERB         10,817 ( 42.2%)
    NOUN          8,386 ( 32.7%)
    ADJ           2,797 ( 10.9%)
    ADV           2,066 (  8.1%)
    X             1,168 (  4.6%)
    PRON            14

## Cell 3: English PoS Tagging with spaCy (UD tagset)

In [4]:
"""
Cell 3: English PoS Tagging with spaCy (UD tagset)
====================================================
Tags all English sentences in the small parallel corpus.
Uses spaCy's en_core_web_sm model which maps to Universal Dependencies tags.

Prerequisites:
  pip install spacy
  python -m spacy download en_core_web_sm

If en_core_web_sm is not installed, run this in a cell first:
  !pip install spacy
  !python -m spacy download en_core_web_sm
"""

import os
import json
import time
import spacy

# ============================================================
# PATHS - UPDATE AS NEEDED
# ============================================================
DATA_DIR = "data/"   # Where your original files are
OUTPUT_DIR = "data2/"  # Where to save outputs

small_en_path = os.path.join(DATA_DIR, "small.en")
small_mz_path = os.path.join(DATA_DIR, "small.mz")

# ============================================================
# Load spaCy English model
# ============================================================
print("Loading spaCy English model...")
try:
    nlp = spacy.load("en_core_web_sm")
    print(f"  ✓ Loaded: en_core_web_sm (pipeline: {nlp.pipe_names})")
except OSError:
    print("  ✗ Model not found. Please run:")
    print("    !pip install spacy")
    print("    !python -m spacy download en_core_web_sm")
    raise

# ============================================================
# Load sentences
# ============================================================
def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
small_mz = read_lines(small_mz_path)
print(f"  Loaded {len(small_en):,} English sentences")
print(f"  Loaded {len(small_mz):,} Mizo sentences")

# ============================================================
# Tag English sentences using spaCy
# spaCy's .pos_ attribute gives Universal Dependencies tags
# ============================================================
print(f"\nTagging {len(small_en):,} English sentences...")
print("  (This may take 1-3 minutes on your laptop)")

start_time = time.time()

tagged_english = []  # List of lists: [[{"token": "word", "pos": "TAG", "lemma": "..."}, ...], ...]

# Use nlp.pipe for efficient batch processing
batch_size = 500
processed = 0

for doc in nlp.pipe(small_en, batch_size=batch_size, n_process=1):
    sent_tokens = []
    for token in doc:
        sent_tokens.append({
            "token": token.text,
            "pos": token.pos_,       # UD tag (NOUN, VERB, etc.)
            "tag": token.tag_,       # Fine-grained tag (NN, VBZ, etc.)
            "lemma": token.lemma_,
            "dep": token.dep_,       # Dependency relation
            "idx": token.idx,        # Character offset
            "is_punct": token.is_punct,
            "is_stop": token.is_stop,
        })
    tagged_english.append(sent_tokens)
    processed += 1
    if processed % 2000 == 0:
        elapsed = time.time() - start_time
        rate = processed / elapsed
        remaining = (len(small_en) - processed) / rate
        print(f"    Processed {processed:>6,}/{len(small_en):,} "
              f"({100*processed/len(small_en):.1f}%) "
              f"[{elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining]")

elapsed = time.time() - start_time
print(f"\n  ✓ Tagging complete in {elapsed:.1f}s ({len(small_en)/elapsed:.0f} sentences/sec)")

# ============================================================
# Verify tagging quality
# ============================================================
print("\n" + "=" * 60)
print("TAGGING VERIFICATION")
print("=" * 60)

# Tag distribution across all tokens
from collections import Counter
all_pos_tags = Counter()
total_tokens = 0
for sent in tagged_english:
    for tok in sent:
        all_pos_tags[tok["pos"]] += 1
        total_tokens += 1

print(f"\n  Total tokens tagged: {total_tokens:,}")
print(f"  Unique UD tags: {len(all_pos_tags)}")
print(f"\n  Tag distribution:")
for tag, count in all_pos_tags.most_common():
    print(f"    {tag:<10} {count:>10,} ({100*count/total_tokens:>5.1f}%)")

# Show sample tagged sentences
print(f"\n  --- Sample Tagged Sentences (first 5) ---")
for i in range(min(5, len(tagged_english))):
    en_sent = small_en[i]
    mz_sent = small_mz[i]
    tokens_str = " ".join([f"{t['token']}/{t['pos']}" for t in tagged_english[i]])
    print(f"\n  [{i}] EN: {en_sent}")
    print(f"      MZ: {mz_sent}")
    print(f"      Tagged: {tokens_str}")

# ============================================================
# Check sentence length alignment (EN tokens vs MZ tokens)
# ============================================================
print(f"\n  --- Sentence Length Comparison ---")
en_lens = [len(sent) for sent in tagged_english]
mz_lens = [len(s.split()) for s in small_mz]

import statistics
print(f"    EN tokens/sent: mean={statistics.mean(en_lens):.1f}, median={statistics.median(en_lens):.0f}")
print(f"    MZ tokens/sent: mean={statistics.mean(mz_lens):.1f}, median={statistics.median(mz_lens):.0f}")

# Length ratio analysis (important for alignment quality)
ratios = [mz/en if en > 0 else 0 for mz, en in zip(mz_lens, en_lens)]
print(f"    MZ/EN length ratio: mean={statistics.mean(ratios):.2f}, median={statistics.median(ratios):.2f}")

# ============================================================
# Export tagged English data
# ============================================================
print("\n" + "=" * 60)
print("EXPORTING TAGGED DATA")
print("=" * 60)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Export as JSON (full information, for programmatic use)
json_path = os.path.join(OUTPUT_DIR, "small_en_tagged.json")
export_data = []
for i in range(len(tagged_english)):
    export_data.append({
        "idx": i,
        "en_text": small_en[i],
        "mz_text": small_mz[i],
        "en_tokens": tagged_english[i]
    })

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=None)
    # Using indent=None to keep file size manageable

file_size = os.path.getsize(json_path) / (1024 * 1024)
print(f"  ✓ Exported: {json_path} ({file_size:.2f} MB)")

# 2. Export as simple token/tag format (one sentence per line, token_tag pairs)
simple_path = os.path.join(OUTPUT_DIR, "small_en_tagged.txt")
with open(simple_path, "w", encoding="utf-8") as f:
    for sent in tagged_english:
        line = " ".join([f"{t['token']}/{t['pos']}" for t in sent])
        f.write(line + "\n")

file_size = os.path.getsize(simple_path) / (1024 * 1024)
print(f"  ✓ Exported: {simple_path} ({file_size:.2f} MB)")

# 3. Export just the UD tags (one sentence per line, space-separated)
tags_path = os.path.join(OUTPUT_DIR, "small_en_tags_only.txt")
with open(tags_path, "w", encoding="utf-8") as f:
    for sent in tagged_english:
        line = " ".join([t["pos"] for t in sent])
        f.write(line + "\n")

file_size = os.path.getsize(tags_path) / (1024 * 1024)
print(f"  ✓ Exported: {tags_path} ({file_size:.2f} MB)")

print(f"\n  Total sentences exported: {len(tagged_english):,}")

print("\n" + "=" * 60)
print("Cell 3 Complete. Please report results before proceeding.")
print("=" * 60)
print("\nNext: Cell 4 will perform word alignment (Mizo ↔ English)")
print("using eflomal or awesome-align for tag projection.")

Loading spaCy English model...
  ✓ Loaded: en_core_web_sm (pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner'])
  Loaded 13,155 English sentences
  Loaded 13,155 Mizo sentences

Tagging 13,155 English sentences...
  (This may take 1-3 minutes on your laptop)
    Processed  2,000/13,155 (15.2%) [5.8s elapsed, ~32.5s remaining]
    Processed  4,000/13,155 (30.4%) [11.0s elapsed, ~25.1s remaining]
    Processed  6,000/13,155 (45.6%) [15.8s elapsed, ~18.8s remaining]
    Processed  8,000/13,155 (60.8%) [20.6s elapsed, ~13.3s remaining]
    Processed 10,000/13,155 (76.0%) [25.5s elapsed, ~8.0s remaining]
    Processed 12,000/13,155 (91.2%) [30.3s elapsed, ~2.9s remaining]

  ✓ Tagging complete in 33.6s (392 sentences/sec)

TAGGING VERIFICATION

  Total tokens tagged: 164,995
  Unique UD tags: 17

  Tag distribution:
    PRON           23,358 ( 14.2%)
    VERB           22,365 ( 13.6%)
    NOUN           22,268 ( 13.5%)
    PUNCT          21,175 ( 12.8%)
    AUX

## Cell 4: Word Alignment (English ↔ Mizo) using SimAlign

In [6]:
"""
Cell 4: Word Alignment (English ↔ Mizo) using SimAlign
========================================================
SimAlign uses multilingual sentence embeddings (mBERT/XLM-R) 
to produce word alignments without training.

Prerequisites:
  pip install simalign

This will also download a multilingual model (~700MB) on first run.
"""

import os
import json
import time
import pickle
from collections import Counter

# ============================================================
# PATHS
# ============================================================
DATA_DIR = "data"
OUTPUT_DIR = "data2"

small_en_path = os.path.join(DATA_DIR, "small.en")
small_mz_path = os.path.join(DATA_DIR, "small.mz")
tagged_json_path = os.path.join(OUTPUT_DIR, "small_en_tagged.json")

# ============================================================
# Install simalign if needed
# ============================================================
try:
    from simalign import SentenceAligner
    print("✓ simalign is available")
except ImportError:
    print("Installing simalign...")
    import subprocess
    subprocess.check_call(["pip", "install", "simalign"])
    from simalign import SentenceAligner
    print("✓ simalign installed and imported")

# ============================================================
# Load data
# ============================================================
print("\nLoading data...")

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
small_mz = read_lines(small_mz_path)

with open(tagged_json_path, "r", encoding="utf-8") as f:
    tagged_data = json.load(f)

print(f"  Loaded {len(small_en):,} sentence pairs")
print(f"  Loaded {len(tagged_data):,} tagged English sentences")

# ============================================================
# Initialize SimAlign
# Uses mBERT (multilingual BERT) by default
# Methods: "itermax" generally gives best quality
# ============================================================
print("\nInitializing SimAlign (downloading model on first run ~700MB)...")
print("  Using mBERT (bert-base-multilingual-cased)")

aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")
# "mai" = max-match + itermax + argmax (we'll use itermax results)
print("  ✓ SimAlign initialized")

# ============================================================
# Run alignment on small corpus
# ============================================================
print(f"\nAligning {len(small_en):,} sentence pairs...")
print("  This may take 15-30 minutes on your GPU (RTX 3050)")
print("  Using GPU if available, otherwise CPU")

start_time = time.time()
alignments = []  # List of alignment dicts per sentence
errors = []

for i in range(len(small_en)):
    try:
        en_tokens = small_en[i].split()
        mz_tokens = small_mz[i].split()
        
        if len(en_tokens) == 0 or len(mz_tokens) == 0:
            alignments.append({"itermax": [], "error": "empty_sentence"})
            continue
        
        # Get alignments - returns dict with keys for each method
        result = aligner.get_word_aligns(en_tokens, mz_tokens)
        
        # result has keys like 'mwmf', 'inter', 'itermax'
        # itermax is generally the best quality
        alignments.append({
            "itermax": result.get("itermax", []),
            "inter": result.get("inter", []),
            "mwmf": result.get("mwmf", []),
        })
        
    except Exception as e:
        alignments.append({"itermax": [], "error": str(e)})
        errors.append((i, str(e)))
    
    if (i + 1) % 500 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(small_en) - (i + 1)) / rate
        print(f"    Aligned {i+1:>6,}/{len(small_en):,} "
              f"({100*(i+1)/len(small_en):.1f}%) "
              f"[{elapsed:.1f}s elapsed, ~{remaining:.0f}s remaining]")

elapsed = time.time() - start_time
print(f"\n  ✓ Alignment complete in {elapsed:.1f}s ({len(small_en)/elapsed:.1f} pairs/sec)")
if errors:
    print(f"  ⚠️  {len(errors)} errors encountered")
    for idx, err in errors[:5]:
        print(f"    Sentence {idx}: {err}")

# ============================================================
# Analyze alignment quality
# ============================================================
print("\n" + "=" * 60)
print("ALIGNMENT QUALITY ANALYSIS")
print("=" * 60)

# Count alignments per sentence
align_counts = [len(a.get("itermax", [])) for a in alignments]
en_token_counts = [len(s.split()) for s in small_en]
mz_token_counts = [len(s.split()) for s in small_mz]

# Coverage: what fraction of EN tokens got aligned?
en_aligned_counts = []
mz_aligned_counts = []
for i, a in enumerate(alignments):
    pairs = a.get("itermax", [])
    en_aligned = len(set(p[0] for p in pairs))
    mz_aligned = len(set(p[1] for p in pairs))
    en_aligned_counts.append(en_aligned)
    mz_aligned_counts.append(mz_aligned)

import statistics
en_coverages = [ea / et if et > 0 else 0 for ea, et in zip(en_aligned_counts, en_token_counts)]
mz_coverages = [ma / mt if mt > 0 else 0 for ma, mt in zip(mz_aligned_counts, mz_token_counts)]

print(f"\n  Alignment statistics (itermax method):")
print(f"    Alignments/sentence: mean={statistics.mean(align_counts):.1f}, "
      f"median={statistics.median(align_counts):.0f}")
print(f"    EN token coverage:   mean={statistics.mean(en_coverages):.2%}, "
      f"median={statistics.median(en_coverages):.2%}")
print(f"    MZ token coverage:   mean={statistics.mean(mz_coverages):.2%}, "
      f"median={statistics.median(mz_coverages):.2%}")

# Distribution of coverage
low_coverage = sum(1 for c in en_coverages if c < 0.3)
mid_coverage = sum(1 for c in en_coverages if 0.3 <= c < 0.7)
high_coverage = sum(1 for c in en_coverages if c >= 0.7)
print(f"\n  EN coverage distribution:")
print(f"    Low  (<30%):  {low_coverage:>6,} ({100*low_coverage/len(en_coverages):.1f}%)")
print(f"    Mid  (30-70%): {mid_coverage:>5,} ({100*mid_coverage/len(en_coverages):.1f}%)")
print(f"    High (>70%):  {high_coverage:>6,} ({100*high_coverage/len(en_coverages):.1f}%)")

# ============================================================
# Show sample alignments
# ============================================================
print(f"\n  --- Sample Alignments (first 5) ---")
for i in range(min(5, len(alignments))):
    en_tokens = small_en[i].split()
    mz_tokens = small_mz[i].split()
    pairs = alignments[i].get("itermax", [])
    
    print(f"\n  [{i}] EN: {small_en[i][:90]}")
    print(f"      MZ: {small_mz[i][:90]}")
    print(f"      Alignments ({len(pairs)} pairs):")
    for en_idx, mz_idx in pairs[:15]:  # Show first 15 pairs
        en_word = en_tokens[en_idx] if en_idx < len(en_tokens) else "?"
        mz_word = mz_tokens[mz_idx] if mz_idx < len(mz_tokens) else "?"
        print(f"        EN[{en_idx}] '{en_word}' ↔ MZ[{mz_idx}] '{mz_word}'")
    if len(pairs) > 15:
        print(f"        ... and {len(pairs)-15} more")

# ============================================================
# Export alignments
# ============================================================
print("\n" + "=" * 60)
print("EXPORTING ALIGNMENTS")
print("=" * 60)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Export as pickle (fast to reload, preserves structure)
pickle_path = os.path.join(OUTPUT_DIR, "small_alignments.pkl")
with open(pickle_path, "wb") as f:
    pickle.dump(alignments, f)
file_size = os.path.getsize(pickle_path) / (1024 * 1024)
print(f"  ✓ Exported: {pickle_path} ({file_size:.2f} MB)")

# 2. Export as JSON (human-readable)
json_align_path = os.path.join(OUTPUT_DIR, "small_alignments.json")
with open(json_align_path, "w", encoding="utf-8") as f:
    json.dump(alignments, f, ensure_ascii=False)
file_size = os.path.getsize(json_align_path) / (1024 * 1024)
print(f"  ✓ Exported: {json_align_path} ({file_size:.2f} MB)")

# 3. Export in Pharaoh format (standard: "0-1 1-3 2-0 ...")
# This is the standard format used by MT tools
pharaoh_path = os.path.join(OUTPUT_DIR, "small_alignments_pharaoh.txt")
with open(pharaoh_path, "w", encoding="utf-8") as f:
    for a in alignments:
        pairs = a.get("itermax", [])
        line = " ".join([f"{en_idx}-{mz_idx}" for en_idx, mz_idx in pairs])
        f.write(line + "\n")
file_size = os.path.getsize(pharaoh_path) / (1024 * 1024)
print(f"  ✓ Exported: {pharaoh_path} ({file_size:.2f} MB)")

print(f"\n  Total alignments exported: {len(alignments):,}")

print("\n" + "=" * 60)
print("Cell 4 Complete. Please report results before proceeding.")
print("=" * 60)
print("\nNext: Cell 5 will project English PoS tags onto Mizo tokens")
print("using the alignments, then verify against the Mizo lexicon.")

✓ simalign is available

Loading data...
  Loaded 13,155 sentence pairs
  Loaded 13,155 tagged English sentences

Initializing SimAlign (downloading model on first run ~700MB)...
  Using mBERT (bert-base-multilingual-cased)


2026-02-17 13:09:07,507 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


  ✓ SimAlign initialized

Aligning 13,155 sentence pairs...
  This may take 15-30 minutes on your GPU (RTX 3050)
  Using GPU if available, otherwise CPU
    Aligned    500/13,155 (3.8%) [60.2s elapsed, ~1524s remaining]
    Aligned  1,000/13,155 (7.6%) [105.7s elapsed, ~1285s remaining]
    Aligned  1,500/13,155 (11.4%) [154.9s elapsed, ~1204s remaining]
    Aligned  2,000/13,155 (15.2%) [204.4s elapsed, ~1140s remaining]
    Aligned  2,500/13,155 (19.0%) [254.6s elapsed, ~1085s remaining]
    Aligned  3,000/13,155 (22.8%) [300.3s elapsed, ~1017s remaining]
    Aligned  3,500/13,155 (26.6%) [344.1s elapsed, ~949s remaining]
    Aligned  4,000/13,155 (30.4%) [388.8s elapsed, ~890s remaining]
    Aligned  4,500/13,155 (34.2%) [436.8s elapsed, ~840s remaining]
    Aligned  5,000/13,155 (38.0%) [470.9s elapsed, ~768s remaining]
    Aligned  5,500/13,155 (41.8%) [512.5s elapsed, ~713s remaining]
    Aligned  6,000/13,155 (45.6%) [558.1s elapsed, ~665s remaining]
    Aligned  6,500/13,155 (4

## Cell 5: Tag Projection + Lexicon Verification

In [1]:
"""
Cell 5: PoS Tag Projection + Lexicon Verification
====================================================
Projects English UD tags onto Mizo tokens via word alignments,
then verifies/corrects/fills gaps using the Mizo lexicon.

Strategy:
1. For each aligned pair (EN[i] ↔ MZ[j]), assign EN[i]'s UD tag to MZ[j]
2. If a MZ token gets multiple tags from different EN alignments, use majority vote
3. Verify projected tags against the Mizo lexicon
4. For unaligned MZ tokens, attempt to assign tags from the lexicon
5. Compute confidence scores for each tag assignment
"""

import os
import json
import pickle
import time
from collections import Counter, defaultdict

# ============================================================
# PATHS
# ============================================================
DATA_DIR = "data"
OUTPUT_DIR = "data2"

small_en_path = os.path.join(DATA_DIR, "small.en")
small_mz_path = os.path.join(DATA_DIR, "small.mz")
tagged_json_path = os.path.join(OUTPUT_DIR, "small_en_tagged.json")
alignments_pkl_path = os.path.join(OUTPUT_DIR, "small_alignments.pkl")
words_clean_path = os.path.join(OUTPUT_DIR, "mizo_words_clean.tsv")
phrases_clean_path = os.path.join(OUTPUT_DIR, "mizo_phrases_clean.tsv")

# ============================================================
# Load all data
# ============================================================
print("Loading data...")

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
small_mz = read_lines(small_mz_path)

with open(tagged_json_path, "r", encoding="utf-8") as f:
    tagged_data = json.load(f)

with open(alignments_pkl_path, "rb") as f:
    alignments = pickle.load(f)

print(f"  Loaded {len(small_en):,} sentence pairs")
print(f"  Loaded {len(tagged_data):,} tagged English sentences")
print(f"  Loaded {len(alignments):,} alignments")

# ============================================================
# Build Mizo lexicon lookup
# ============================================================
print("\nBuilding Mizo lexicon lookup...")

# Word lexicon: word -> list of possible tags
word_lexicon = defaultdict(list)
with open(words_clean_path, "r", encoding="utf-8") as f:
    next(f)  # skip header
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            word, tag = parts
            word_lexicon[word.lower()].append(tag)

# Deduplicate tags per word but keep frequency info
word_lexicon_tags = {}
for word, tags in word_lexicon.items():
    tag_counts = Counter(tags)
    word_lexicon_tags[word] = tag_counts

# Phrase lexicon
phrase_lexicon = defaultdict(list)
with open(phrases_clean_path, "r", encoding="utf-8") as f:
    next(f)  # skip header
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            phrase, tag = parts
            phrase_lexicon[phrase.lower()].append(tag)

phrase_lexicon_tags = {}
for phrase, tags in phrase_lexicon.items():
    tag_counts = Counter(tags)
    phrase_lexicon_tags[phrase] = tag_counts

print(f"  Word lexicon: {len(word_lexicon_tags):,} unique words")
print(f"  Phrase lexicon: {len(phrase_lexicon_tags):,} unique phrases")

# ============================================================
# Helper: Check if a token is punctuation
# ============================================================
import re
def is_punctuation(token):
    return bool(re.match(r'^[^\w\s]+$', token)) or token in {
        '.', ',', '!', '?', ';', ':', '"', "'", '(', ')', '[', ']',
        '{', '}', '-', '–', '—', '...', '/', '\\', '&', '*', '#',
        '@', '%', '+', '=', '<', '>', '~', '^', '|', '`'
    }

# ============================================================
# Step 1: Project English tags onto Mizo via alignments
# ============================================================
print("\n" + "=" * 60)
print("STEP 1: TAG PROJECTION")
print("=" * 60)

projected_corpus = []  # List of sentences, each sentence = list of token dicts

for sent_idx in range(len(small_mz)):
    mz_tokens = small_mz[sent_idx].split()
    en_tagged = tagged_data[sent_idx]["en_tokens"]
    align_pairs = alignments[sent_idx].get("itermax", [])
    
    # Initialize Mizo token entries
    mz_sent = []
    for tok_idx, token in enumerate(mz_tokens):
        mz_sent.append({
            "token": token,
            "idx": tok_idx,
            "projected_tags": [],      # Tags from EN projection
            "lexicon_tags": [],        # Tags from lexicon lookup
            "final_tag": None,         # Resolved tag
            "confidence": 0.0,         # Confidence score
            "source": "untagged",      # How tag was determined
        })
    
    # Project: for each alignment pair, copy EN tag to MZ token
    for en_idx, mz_idx in align_pairs:
        if en_idx < len(en_tagged) and mz_idx < len(mz_sent):
            en_tag = en_tagged[en_idx]["pos"]
            mz_sent[mz_idx]["projected_tags"].append(en_tag)
    
    projected_corpus.append(mz_sent)

# Count projection coverage
total_mz_tokens = sum(len(sent) for sent in projected_corpus)
projected_tokens = sum(1 for sent in projected_corpus for tok in sent if tok["projected_tags"])
print(f"\n  Total MZ tokens: {total_mz_tokens:,}")
print(f"  Tokens with projected tags: {projected_tokens:,} ({100*projected_tokens/total_mz_tokens:.1f}%)")

# ============================================================
# Step 2: Lexicon lookup for all Mizo tokens
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: LEXICON LOOKUP")
print("=" * 60)

lexicon_hits = 0
for sent in projected_corpus:
    for tok in sent:
        token_lower = tok["token"].lower().strip(".,!?;:'\"()-")
        if token_lower in word_lexicon_tags:
            tok["lexicon_tags"] = list(word_lexicon_tags[token_lower].keys())
            lexicon_hits += 1

print(f"  Tokens found in word lexicon: {lexicon_hits:,} ({100*lexicon_hits/total_mz_tokens:.1f}%)")

# Phrase lexicon matching (check bigrams, trigrams, etc.)
phrase_hits = 0
for sent in projected_corpus:
    tokens_lower = [t["token"].lower().strip(".,!?;:'\"()-") for t in sent]
    # Check phrases of length 2-5
    for phrase_len in range(2, 6):
        for start in range(len(tokens_lower) - phrase_len + 1):
            phrase = " ".join(tokens_lower[start:start + phrase_len])
            if phrase in phrase_lexicon_tags:
                phrase_hits += 1
                # Assign the phrase tag to all tokens in the phrase
                phrase_tag = phrase_lexicon_tags[phrase].most_common(1)[0][0]
                for j in range(start, start + phrase_len):
                    if phrase_tag not in sent[j]["lexicon_tags"]:
                        sent[j]["lexicon_tags"].append(phrase_tag)

print(f"  Phrase matches found: {phrase_hits:,}")

# ============================================================
# Step 3: Resolve final tags with confidence scoring
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: TAG RESOLUTION")
print("=" * 60)

"""
Resolution strategy (priority order):
1. PUNCT: If token is punctuation → PUNCT (confidence 1.0)
2. AGREEMENT: Projected tag matches lexicon tag → use it (confidence 0.95)
3. LEXICON_OVERRIDE: Projected tag conflicts with lexicon → use lexicon majority (confidence 0.80)
4. PROJECTION_ONLY: No lexicon entry, use projected majority tag (confidence 0.60)
5. LEXICON_ONLY: No projection, use lexicon majority tag (confidence 0.70)
6. UNTAGGED: No information available (confidence 0.0)
"""

resolution_stats = Counter()

for sent in projected_corpus:
    for tok in sent:
        token = tok["token"]
        projected = tok["projected_tags"]
        lexicon = tok["lexicon_tags"]
        
        # Case 1: Punctuation
        if is_punctuation(token.strip()):
            tok["final_tag"] = "PUNCT"
            tok["confidence"] = 1.0
            tok["source"] = "punct_rule"
            resolution_stats["punct_rule"] += 1
            continue
        
        projected_counter = Counter(projected)
        
        if projected and lexicon:
            # Check for agreement
            projected_best = projected_counter.most_common(1)[0][0]
            if projected_best in lexicon:
                # Case 2: Agreement
                tok["final_tag"] = projected_best
                tok["confidence"] = 0.95
                tok["source"] = "agreement"
                resolution_stats["agreement"] += 1
            else:
                # Case 3: Conflict - trust lexicon (it's human-verified)
                # But use projected tag frequency as secondary signal
                # Pick lexicon tag that appears most in projected tags, or most common lexicon tag
                best_lexicon_tag = None
                best_score = -1
                for lt in lexicon:
                    score = projected_counter.get(lt, 0)
                    if score > best_score:
                        best_score = score
                        best_lexicon_tag = lt
                
                if best_lexicon_tag and best_score > 0:
                    tok["final_tag"] = best_lexicon_tag
                    tok["confidence"] = 0.85
                    tok["source"] = "lexicon_partial_agree"
                    resolution_stats["lexicon_partial_agree"] += 1
                else:
                    # No overlap at all - prefer lexicon
                    token_lower = token.lower().strip(".,!?;:'\"()-")
                    if token_lower in word_lexicon_tags:
                        tok["final_tag"] = word_lexicon_tags[token_lower].most_common(1)[0][0]
                    else:
                        tok["final_tag"] = lexicon[0]  # First lexicon tag
                    tok["confidence"] = 0.75
                    tok["source"] = "lexicon_override"
                    resolution_stats["lexicon_override"] += 1
        
        elif projected and not lexicon:
            # Case 4: Projection only
            tok["final_tag"] = projected_counter.most_common(1)[0][0]
            tok["confidence"] = 0.60
            tok["source"] = "projection_only"
            resolution_stats["projection_only"] += 1
        
        elif not projected and lexicon:
            # Case 5: Lexicon only
            token_lower = token.lower().strip(".,!?;:'\"()-")
            if token_lower in word_lexicon_tags:
                tok["final_tag"] = word_lexicon_tags[token_lower].most_common(1)[0][0]
            else:
                tok["final_tag"] = lexicon[0]
            tok["confidence"] = 0.70
            tok["source"] = "lexicon_only"
            resolution_stats["lexicon_only"] += 1
        
        else:
            # Case 6: No information
            tok["final_tag"] = None
            tok["confidence"] = 0.0
            tok["source"] = "untagged"
            resolution_stats["untagged"] += 1

# ============================================================
# Statistics
# ============================================================
print(f"\n  Resolution statistics:")
for source, count in resolution_stats.most_common():
    print(f"    {source:<25} {count:>8,} ({100*count/total_mz_tokens:>5.1f}%)")

tagged_tokens = sum(1 for sent in projected_corpus for tok in sent if tok["final_tag"] is not None)
print(f"\n  Total tokens tagged: {tagged_tokens:,}/{total_mz_tokens:,} ({100*tagged_tokens/total_mz_tokens:.1f}%)")

untagged_tokens = total_mz_tokens - tagged_tokens
print(f"  Untagged tokens:     {untagged_tokens:,} ({100*untagged_tokens/total_mz_tokens:.1f}%)")

# Confidence distribution
confidences = [tok["confidence"] for sent in projected_corpus for tok in sent if tok["final_tag"]]
if confidences:
    import statistics
    print(f"\n  Confidence scores (tagged tokens only):")
    print(f"    Mean:   {statistics.mean(confidences):.3f}")
    print(f"    Median: {statistics.median(confidences):.3f}")
    print(f"    >=0.90: {sum(1 for c in confidences if c >= 0.90):,} ({100*sum(1 for c in confidences if c >= 0.90)/len(confidences):.1f}%)")
    print(f"    >=0.70: {sum(1 for c in confidences if c >= 0.70):,} ({100*sum(1 for c in confidences if c >= 0.70)/len(confidences):.1f}%)")
    print(f"    >=0.50: {sum(1 for c in confidences if c >= 0.50):,} ({100*sum(1 for c in confidences if c >= 0.50)/len(confidences):.1f}%)")

# Final tag distribution
final_tag_dist = Counter()
for sent in projected_corpus:
    for tok in sent:
        if tok["final_tag"]:
            final_tag_dist[tok["final_tag"]] += 1

print(f"\n  Final tag distribution (Mizo):")
for tag, count in final_tag_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/tagged_tokens:>5.1f}%)")

# ============================================================
# Show sample tagged Mizo sentences
# ============================================================
print("\n" + "=" * 60)
print("SAMPLE TAGGED MIZO SENTENCES")
print("=" * 60)

for i in range(min(10, len(projected_corpus))):
    mz_sent = projected_corpus[i]
    en_sent = small_en[i]
    
    tagged_str = " ".join([
        f"{t['token']}/{t['final_tag'] or '?'}[{t['source'][:3]}]"
        for t in mz_sent
    ])
    
    # Compact version
    compact = " ".join([
        f"{t['token']}/{t['final_tag'] or '?'}"
        for t in mz_sent
    ])
    
    print(f"\n  [{i}] EN: {en_sent[:100]}")
    print(f"      MZ: {compact}")

# ============================================================
# Export projected/tagged Mizo data
# ============================================================
print("\n" + "=" * 60)
print("EXPORTING TAGGED MIZO DATA")
print("=" * 60)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Full JSON with all metadata
full_json_path = os.path.join(OUTPUT_DIR, "small_mz_tagged_full.json")
with open(full_json_path, "w", encoding="utf-8") as f:
    json.dump(projected_corpus, f, ensure_ascii=False)
file_size = os.path.getsize(full_json_path) / (1024 * 1024)
print(f"  ✓ {full_json_path} ({file_size:.2f} MB)")

# 2. CoNLL-style format (token\ttag per line, blank line between sentences)
conll_path = os.path.join(OUTPUT_DIR, "small_mz_tagged.conll")
with open(conll_path, "w", encoding="utf-8") as f:
    for sent in projected_corpus:
        for tok in sent:
            tag = tok["final_tag"] or "X"
            f.write(f"{tok['token']}\t{tag}\t{tok['confidence']:.2f}\t{tok['source']}\n")
        f.write("\n")
file_size = os.path.getsize(conll_path) / (1024 * 1024)
print(f"  ✓ {conll_path} ({file_size:.2f} MB)")

# 3. Simple token/tag format (one sentence per line)
simple_path = os.path.join(OUTPUT_DIR, "small_mz_tagged_simple.txt")
with open(simple_path, "w", encoding="utf-8") as f:
    for sent in projected_corpus:
        tokens = " ".join([f"{t['token']}/{t['final_tag'] or 'X'}" for t in sent])
        f.write(tokens + "\n")
file_size = os.path.getsize(simple_path) / (1024 * 1024)
print(f"  ✓ {simple_path} ({file_size:.2f} MB)")

# 4. High-confidence subset only (confidence >= 0.70 for ALL tokens in sentence)
hc_conll_path = os.path.join(OUTPUT_DIR, "small_mz_tagged_highconf.conll")
hc_count = 0
with open(hc_conll_path, "w", encoding="utf-8") as f:
    for sent in projected_corpus:
        # Check if ALL tokens have tag and confidence >= 0.60
        all_tagged = all(tok["final_tag"] is not None for tok in sent)
        min_conf = min(tok["confidence"] for tok in sent) if sent else 0
        if all_tagged and min_conf >= 0.60:
            hc_count += 1
            for tok in sent:
                f.write(f"{tok['token']}\t{tok['final_tag']}\n")
            f.write("\n")

file_size = os.path.getsize(hc_conll_path) / (1024 * 1024)
print(f"  ✓ {hc_conll_path} ({file_size:.2f} MB) — {hc_count:,} high-confidence sentences")

# 5. Pickle for fast reloading
pkl_path = os.path.join(OUTPUT_DIR, "small_mz_projected.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
file_size = os.path.getsize(pkl_path) / (1024 * 1024)
print(f"  ✓ {pkl_path} ({file_size:.2f} MB)")

print(f"\n  Summary:")
print(f"    Total sentences:           {len(projected_corpus):>8,}")
print(f"    High-confidence sentences: {hc_count:>8,} ({100*hc_count/len(projected_corpus):.1f}%)")
print(f"    Total tokens tagged:       {tagged_tokens:>8,}/{total_mz_tokens:,}")

print("\n" + "=" * 60)
print("Cell 5 Complete. Please report results before proceeding.")
print("=" * 60)
print("\nNext: Cell 6 will handle untagged tokens and prepare")
print("the final training dataset. Then Cell 7+ will process")
print("the large corpus for additional training data.")

Loading data...
  Loaded 13,155 sentence pairs
  Loaded 13,155 tagged English sentences
  Loaded 13,155 alignments

Building Mizo lexicon lookup...
  Word lexicon: 44,479 unique words
  Phrase lexicon: 22,924 unique phrases

STEP 1: TAG PROJECTION

  Total MZ tokens: 157,146
  Tokens with projected tags: 75,296 (47.9%)

STEP 2: LEXICON LOOKUP
  Tokens found in word lexicon: 146,471 (93.2%)
  Phrase matches found: 8,493

STEP 3: TAG RESOLUTION

  Resolution statistics:
    lexicon_only                78,938 ( 50.2%)
    lexicon_override            47,126 ( 30.0%)
    agreement                   19,303 ( 12.3%)
    projection_only              7,469 (  4.8%)
    untagged                     2,904 (  1.8%)
    lexicon_partial_agree        1,363 (  0.9%)
    punct_rule                      43 (  0.0%)

  Total tokens tagged: 154,242/157,146 (98.2%)
  Untagged tokens:     2,904 (1.8%)

  Confidence scores (tagged tokens only):
    Mean:   0.743
    Median: 0.700
    >=0.90: 19,346 (12.5%)
 

## Cell 6: Handle Untagged Tokens + Prepare Training Data

In [2]:
"""
Cell 6: Handle Untagged Tokens + Prepare Training Data
========================================================
1. Analyze untagged tokens and apply heuristic rules
2. Build final training-ready dataset
3. Create train/dev/test splits
4. Export in standard CoNLL format for model training
"""

import os
import json
import pickle
import random
import re
from collections import Counter, defaultdict

# ============================================================
# PATHS
# ============================================================
DATA_DIR = "data"
OUTPUT_DIR = "data2"

projected_pkl_path = os.path.join(OUTPUT_DIR, "small_mz_projected.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")
small_mz_path = os.path.join(DATA_DIR, "small.mz")

# ============================================================
# Load data
# ============================================================
print("Loading projected data...")

with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
small_mz = read_lines(small_mz_path)
print(f"  Loaded {len(projected_corpus):,} sentences")

# ============================================================
# Step 1: Analyze and handle untagged tokens
# ============================================================
print("\n" + "=" * 60)
print("STEP 1: ANALYZE UNTAGGED TOKENS")
print("=" * 60)

untagged_tokens = []
for sent_idx, sent in enumerate(projected_corpus):
    for tok in sent:
        if tok["final_tag"] is None:
            untagged_tokens.append({
                "token": tok["token"],
                "sent_idx": sent_idx,
                "context": " ".join([t["token"] for t in sent]),
            })

print(f"\n  Total untagged tokens: {len(untagged_tokens):,}")

# Frequency of untagged tokens
untagged_freq = Counter(t["token"].lower().strip(".,!?;:'\"()-") for t in untagged_tokens)
print(f"  Unique untagged forms: {len(untagged_freq):,}")
print(f"\n  Most common untagged tokens:")
for token, count in untagged_freq.most_common(30):
    print(f"    {token:<30} {count:>5}")

# ============================================================
# Step 2: Apply heuristic rules for untagged tokens
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: HEURISTIC TAGGING OF UNTAGGED TOKENS")
print("=" * 60)

"""
Heuristic rules:
1. Numbers (digits) → NUM
2. Tokens ending with common suffixes → inferred tag
3. Proper nouns (capitalized, not sentence-initial) → PROPN
4. Tokens with punctuation attached → strip and re-check
5. Remaining → X (unknown)
"""

def heuristic_tag(token, tok_idx, sent_tokens):
    """Apply heuristic rules to assign a tag to an untagged token."""
    clean = token.strip(".,!?;:'\"()-!").lower()
    
    # Rule 1: Pure numbers
    if re.match(r'^[\d,]+\.?\d*$', clean):
        return "NUM", "heuristic_num"
    
    # Rule 2: Punctuation (more aggressive check)
    if all(c in '.,!?;:\'"()-–—…/\\&*#@%+=<>~^|`' for c in token):
        return "PUNCT", "heuristic_punct"
    
    # Rule 3: Token is actually punctuation-attached word
    # e.g., "isua" in "isua thlalâk" might be a PROPN (Jesus)
    
    # Rule 4: Capitalized non-initial word → PROPN
    if tok_idx > 0 and token[0].isupper() and len(token) > 1:
        return "PROPN", "heuristic_propn"
    
    # Rule 5: Common Mizo suffixes (morphological heuristics)
    # -na suffix often marks nominalizations (NOUN)
    if clean.endswith('na') and len(clean) > 3:
        return "NOUN", "heuristic_suffix_na"
    # -tu suffix often marks agent nouns (NOUN)
    if clean.endswith('tu') and len(clean) > 3:
        return "NOUN", "heuristic_suffix_tu"
    # -in suffix often marks adverbial/instrumental (ADV)
    if clean.endswith('in') and len(clean) > 3:
        return "ADV", "heuristic_suffix_in"
    
    # Rule 6: Context-based - if surrounded by known tags, use most common neighbor tag
    # (simplified: just assign X)
    
    return "X", "heuristic_unknown"

heuristic_stats = Counter()
for sent_idx, sent in enumerate(projected_corpus):
    for tok_idx, tok in enumerate(sent):
        if tok["final_tag"] is None:
            tag, source = heuristic_tag(tok["token"], tok_idx, sent)
            tok["final_tag"] = tag
            tok["confidence"] = 0.40 if tag != "X" else 0.20
            tok["source"] = source
            heuristic_stats[source] += 1

print(f"\n  Heuristic tagging results:")
for source, count in heuristic_stats.most_common():
    print(f"    {source:<25} {count:>6,}")

# Verify all tokens now have tags
total_tokens = sum(len(sent) for sent in projected_corpus)
tagged_tokens = sum(1 for sent in projected_corpus for tok in sent if tok["final_tag"] is not None)
still_none = total_tokens - tagged_tokens
print(f"\n  All tokens tagged: {tagged_tokens:,}/{total_tokens:,} (remaining None: {still_none})")

# ============================================================
# Step 3: Final quality assessment
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: FINAL QUALITY ASSESSMENT")
print("=" * 60)

# Overall tag distribution
final_dist = Counter()
source_dist = Counter()
for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1
        source_dist[tok["source"]] += 1

print(f"\n  Final tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

print(f"\n  Tag source distribution:")
for source, count in source_dist.most_common():
    print(f"    {source:<25} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

# Confidence distribution
import statistics
all_confs = [tok["confidence"] for sent in projected_corpus for tok in sent]
print(f"\n  Confidence distribution:")
print(f"    Mean:   {statistics.mean(all_confs):.3f}")
print(f"    >=0.90: {sum(1 for c in all_confs if c >= 0.90):>8,}")
print(f"    >=0.70: {sum(1 for c in all_confs if c >= 0.70):>8,}")
print(f"    >=0.50: {sum(1 for c in all_confs if c >= 0.50):>8,}")
print(f"    < 0.50: {sum(1 for c in all_confs if c < 0.50):>8,}")

# ============================================================
# Step 4: Prepare training data with quality-based filtering
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: PREPARE TRAINING DATA")
print("=" * 60)

"""
We'll create multiple quality tiers:
- Tier 1 (HIGH): All tokens have confidence >= 0.70 (most reliable)
- Tier 2 (MEDIUM): All tokens have confidence >= 0.50
- Tier 3 (ALL): All sentences included

For training, we'll primarily use Tier 2 with Tier 1 for dev/test.
"""

tier1_sents = []  # High confidence
tier2_sents = []  # Medium confidence
tier3_sents = []  # All

for sent_idx, sent in enumerate(projected_corpus):
    # Convert to simple format: list of (token, tag) tuples
    token_tags = [(tok["token"], tok["final_tag"]) for tok in sent]
    min_conf = min(tok["confidence"] for tok in sent)
    mean_conf = statistics.mean(tok["confidence"] for tok in sent)
    
    entry = {
        "idx": sent_idx,
        "tokens": token_tags,
        "min_conf": min_conf,
        "mean_conf": mean_conf,
        "en_text": small_en[sent_idx] if sent_idx < len(small_en) else "",
    }
    
    tier3_sents.append(entry)
    if min_conf >= 0.50:
        tier2_sents.append(entry)
    if min_conf >= 0.70:
        tier1_sents.append(entry)

print(f"  Tier 1 (high, min_conf>=0.70): {len(tier1_sents):>8,} sentences")
print(f"  Tier 2 (med,  min_conf>=0.50): {len(tier2_sents):>8,} sentences")
print(f"  Tier 3 (all):                  {len(tier3_sents):>8,} sentences")

# ============================================================
# Step 5: Train / Dev / Test splits
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: TRAIN / DEV / TEST SPLITS")
print("=" * 60)

"""
Strategy:
- Use Tier 1 sentences for dev and test (highest quality for evaluation)
- Use Tier 2 sentences for training (larger dataset, still reasonable quality)
- Split: 80% train, 10% dev, 10% test
- Dev and test are drawn from Tier 1 for reliable evaluation
"""

random.seed(42)

# Shuffle Tier 1 for dev/test selection
tier1_shuffled = tier1_sents.copy()
random.shuffle(tier1_shuffled)

# Reserve some Tier 1 for dev and test
n_dev = max(500, len(tier1_shuffled) // 10)    # At least 500 or 10%
n_test = max(500, len(tier1_shuffled) // 10)   # At least 500 or 10%

# Make sure we have enough
if n_dev + n_test > len(tier1_shuffled):
    n_dev = len(tier1_shuffled) // 3
    n_test = len(tier1_shuffled) // 3

test_set = tier1_shuffled[:n_test]
dev_set = tier1_shuffled[n_test:n_test + n_dev]

# Training: all Tier 2 sentences NOT in dev/test
dev_test_indices = set(s["idx"] for s in dev_set + test_set)
train_set = [s for s in tier2_sents if s["idx"] not in dev_test_indices]

# Also add remaining Tier 1 that wasn't used for dev/test to training
remaining_tier1 = [s for s in tier1_shuffled[n_test + n_dev:] if s["idx"] not in dev_test_indices]
# These are already in train_set via tier2, so no action needed

print(f"  Train: {len(train_set):>8,} sentences")
print(f"  Dev:   {len(dev_set):>8,} sentences")
print(f"  Test:  {len(test_set):>8,} sentences")
print(f"  Total: {len(train_set) + len(dev_set) + len(test_set):>8,} sentences")

# Token counts
train_tokens = sum(len(s["tokens"]) for s in train_set)
dev_tokens = sum(len(s["tokens"]) for s in dev_set)
test_tokens = sum(len(s["tokens"]) for s in test_set)
print(f"\n  Train tokens: {train_tokens:>10,}")
print(f"  Dev tokens:   {dev_tokens:>10,}")
print(f"  Test tokens:  {test_tokens:>10,}")

# Verify tag distributions are similar across splits
print(f"\n  Tag distribution comparison:")
train_tags = Counter(tag for s in train_set for _, tag in s["tokens"])
dev_tags = Counter(tag for s in dev_set for _, tag in s["tokens"])
test_tags = Counter(tag for s in test_set for _, tag in s["tokens"])

all_tags_set = sorted(set(train_tags.keys()) | set(dev_tags.keys()) | set(test_tags.keys()))
print(f"    {'Tag':<10} {'Train%':>8} {'Dev%':>8} {'Test%':>8}")
for tag in all_tags_set:
    t_pct = 100 * train_tags[tag] / train_tokens if train_tokens else 0
    d_pct = 100 * dev_tags[tag] / dev_tokens if dev_tokens else 0
    te_pct = 100 * test_tags[tag] / test_tokens if test_tokens else 0
    print(f"    {tag:<10} {t_pct:>7.1f}% {d_pct:>7.1f}% {te_pct:>7.1f}%")

# ============================================================
# Step 6: Export in CoNLL format
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: EXPORT TRAINING DATA")
print("=" * 60)

os.makedirs(OUTPUT_DIR, exist_ok=True)

def export_conll(sentences, filepath):
    """Export in CoNLL format: token\\ttag per line, blank line between sentences."""
    with open(filepath, "w", encoding="utf-8") as f:
        for sent in sentences:
            for token, tag in sent["tokens"]:
                f.write(f"{token}\t{tag}\n")
            f.write("\n")
    file_size = os.path.getsize(filepath) / (1024 * 1024)
    return file_size

# Main splits
train_path = os.path.join(OUTPUT_DIR, "train.conll")
dev_path = os.path.join(OUTPUT_DIR, "dev.conll")
test_path = os.path.join(OUTPUT_DIR, "test.conll")

sz = export_conll(train_set, train_path)
print(f"  ✓ {train_path} ({sz:.2f} MB, {len(train_set):,} sentences, {train_tokens:,} tokens)")

sz = export_conll(dev_set, dev_path)
print(f"  ✓ {dev_path} ({sz:.2f} MB, {len(dev_set):,} sentences, {dev_tokens:,} tokens)")

sz = export_conll(test_set, test_path)
print(f"  ✓ {test_path} ({sz:.2f} MB, {len(test_set):,} sentences, {test_tokens:,} tokens)")

# Also export all tiers for reference
tier1_path = os.path.join(OUTPUT_DIR, "tier1_highconf.conll")
tier2_path = os.path.join(OUTPUT_DIR, "tier2_medconf.conll")
tier3_path = os.path.join(OUTPUT_DIR, "tier3_all.conll")

sz = export_conll(tier1_sents, tier1_path)
print(f"  ✓ {tier1_path} ({sz:.2f} MB, {len(tier1_sents):,} sentences)")

sz = export_conll(tier2_sents, tier2_path)
print(f"  ✓ {tier2_path} ({sz:.2f} MB, {len(tier2_sents):,} sentences)")

sz = export_conll(tier3_sents, tier3_path)
print(f"  ✓ {tier3_path} ({sz:.2f} MB, {len(tier3_sents):,} sentences)")

# Export metadata/stats as JSON
stats = {
    "total_sentences": len(projected_corpus),
    "total_tokens": total_tokens,
    "tier1_sentences": len(tier1_sents),
    "tier2_sentences": len(tier2_sents),
    "tier3_sentences": len(tier3_sents),
    "train_sentences": len(train_set),
    "dev_sentences": len(dev_set),
    "test_sentences": len(test_set),
    "train_tokens": train_tokens,
    "dev_tokens": dev_tokens,
    "test_tokens": test_tokens,
    "tag_distribution": dict(final_dist.most_common()),
    "source_distribution": dict(source_dist.most_common()),
}
stats_path = os.path.join(OUTPUT_DIR, "dataset_stats.json")
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"  ✓ {stats_path}")

# Save the full projected corpus with all metadata
full_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
with open(full_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {full_pkl_path}")

print("\n" + "=" * 60)
print("Cell 6 Complete. Please report results before proceeding.")
print("=" * 60)
print("\nNext: Cell 7 will process the large corpus (1.3M sentences)")
print("to augment training data. This will take longer but will")
print("significantly improve the tagger's coverage and accuracy.")

Loading projected data...
  Loaded 13,155 sentences

STEP 1: ANALYZE UNTAGGED TOKENS

  Total untagged tokens: 2,904
  Unique untagged forms: 1,020

  Most common untagged tokens:
    duh                              498
    tiin                             126
    nu                               111
    erawh                            108
    isua                              73
    aia                               48
    hetah                             45
    takah                             43
    bei                               30
    nân                               25
    anga                              25
    nia                               24
    zet                               23
    tal                               23
    ruihhlo                           23
    ramah                             23
    bo                                20
    amc                               18
    bu                                16
    ni’                               14


## Cell 6A: Sanity Check - Round 1 (Sentences 1-25)

In [3]:
"""
Cell 6A: Sanity Check - Round 1 (Sentences 1-25)
==================================================
Displays 25 randomly sampled tagged Mizo sentences for human review.

HOW TO REPORT ERRORS:
For each error, note:
  - Sentence number (e.g., S3)
  - The wrong token and its current tag
  - The correct tag

Example error report:
  S3: "chuan" should be PART not NOUN
  S7: "leh" should be CCONJ not VERB
  S12: "a" should be PRON not PART

UD Tags reference:
  ADJ   = Adjective         ADP   = Adposition (postposition)
  ADV   = Adverb            AUX   = Auxiliary verb
  CCONJ = Coordinating conj DET   = Determiner
  INTJ  = Interjection      NOUN  = Noun
  NUM   = Numeral           PART  = Particle
  PRON  = Pronoun            PROPN = Proper noun
  PUNCT = Punctuation        SCONJ = Subordinating conj
  SYM   = Symbol            VERB  = Verb
  X     = Other/Unknown
"""

import os
import pickle
import random

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")

# ============================================================
# Load data
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
print(f"Loaded {len(projected_corpus):,} sentences\n")

# ============================================================
# Sample 25 sentences (use fixed seed for reproducibility)
# We sample from different confidence levels to get a mix
# ============================================================
random.seed(101)  # Round 1 seed

# Get indices with varying quality
all_indices = list(range(len(projected_corpus)))
random.shuffle(all_indices)
sample_indices = sorted(all_indices[:25])

# ============================================================
# Display sentences for review
# ============================================================
print("=" * 70)
print("SANITY CHECK - ROUND 1 (25 sentences)")
print("=" * 70)
print()
print("UD Tags: ADJ ADP ADV AUX CCONJ DET INTJ NOUN NUM")
print("         PART PRON PROPN PUNCT SCONJ SYM VERB X")
print()
print("Please review each sentence and report errors like:")
print('  S3: "chuan" should be PART not NOUN')
print("=" * 70)

for display_num, sent_idx in enumerate(sample_indices, 1):
    sent = projected_corpus[sent_idx]
    en_text = small_en[sent_idx] if sent_idx < len(small_en) else ""
    
    # Build tagged display
    tagged_tokens = []
    for tok in sent:
        tag = tok["final_tag"] or "?"
        conf = tok["confidence"]
        source_short = tok["source"][:3]
        tagged_tokens.append(f"{tok['token']}/{tag}")
    
    # Print with clear formatting
    print(f"\n{'─' * 70}")
    print(f"  S{display_num} (idx={sent_idx})")
    print(f"  EN: {en_text}")
    print(f"  MZ: {' '.join(t['token'] for t in sent)}")
    print(f"  TAG: {' '.join(tagged_tokens)}")
    
    # Also show confidence indicators for low-confidence tokens
    low_conf_tokens = [(tok["token"], tok["final_tag"], tok["confidence"], tok["source"]) 
                       for tok in sent if tok["confidence"] < 0.70]
    if low_conf_tokens:
        print(f"  ⚠️  Low confidence: ", end="")
        print(", ".join([f"'{t}'/{tag} ({conf:.2f},{src})" 
                        for t, tag, conf, src in low_conf_tokens]))

print(f"\n{'─' * 70}")
print(f"\n{'=' * 70}")
print("END OF ROUND 1")
print("=" * 70)
print(f"\nSampled sentence indices: {sample_indices}")
print("\nPlease review and report errors in the format:")
print('  S3: "chuan" should be PART not NOUN')
print('  S7: "leh" should be CCONJ not VERB')
print("\nAfter reporting, we will apply corrections and run Round 2.")

Loaded 13,155 sentences

SANITY CHECK - ROUND 1 (25 sentences)

UD Tags: ADJ ADP ADV AUX CCONJ DET INTJ NOUN NUM
         PART PRON PROPN PUNCT SCONJ SYM VERB X

Please review each sentence and report errors like:
  S3: "chuan" should be PART not NOUN

──────────────────────────────────────────────────────────────────────
  S1 (idx=444)
  EN: A friend told me that story.
  MZ: chu thawnthu chu ka ṭhian pakhatin min hrilh a.
  TAG: chu/VERB thawnthu/ADJ chu/VERB ka/NOUN ṭhian/NOUN pakhatin/PRON min/NOUN hrilh/NOUN a./NOUN

──────────────────────────────────────────────────────────────────────
  S2 (idx=716)
  EN: If we don't work, our bones become stiff.
  MZ: hna kan thawh loh chuan kan ruh te hi a khawng a.
  TAG: hna/NOUN kan/PRON thawh/VERB loh/ADV chuan/NOUN kan/NOUN ruh/NOUN te/VERB hi/ADJ a/PART khawng/VERB a./NOUN

──────────────────────────────────────────────────────────────────────
  S3 (idx=1025)
  EN: I tried hard not to put too much pressure on myself.
  MZ: keimah leh kei

## Cell 6B: Apply Round 1 Corrections + Round 2 Sanity Check

In [4]:
"""
Cell 6B: Apply Round 1 Corrections + Round 2 Sanity Check
============================================================
1. Apply specific corrections from Round 1
2. Apply systematic/global corrections based on observed patterns
3. Re-export corrected data
4. Display Round 2 (15 new sentences)
"""

import os
import pickle
import random
import json
from collections import Counter, defaultdict

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")

# ============================================================
# Load data
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
print(f"Loaded {len(projected_corpus):,} sentences")

# ============================================================
# STEP 1: Define systematic (global) correction rules
# ============================================================
"""
Based on Round 1 review, these tokens are systematically mistagged.
We apply global corrections where the token is almost always a specific tag.

Rules are applied as: if token matches AND current tag is in wrong_tags → change to correct_tag.
Some tokens are context-dependent, so we only correct known wrong assignments.
"""

# Format: token_lower -> {wrong_tags: set, correct_tag: str}
# We strip common trailing punctuation for matching
GLOBAL_CORRECTIONS = {
    # Pronouns frequently mistagged as NOUN
    "ka": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "kan": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "an": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "min": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "i": {"wrong_tags": {"PART"}, "correct_tag": "PRON"},
    "in": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    
    # Conjunctions/particles frequently mistagged
    "chuan": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "leh": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "pawh": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "hian": {"wrong_tags": {"ADJ"}, "correct_tag": "CCONJ"},
    "tih": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    
    # chu: PRON or PART, not VERB
    "chu": {"wrong_tags": {"VERB"}, "correct_tag": "PART"},
    
    # Common function words
    "te": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ten": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ta": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ang": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "awm": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "duh": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "lo": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "nge": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "em": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADV"},
    "vek": {"wrong_tags": {"ADJ"}, "correct_tag": "ADV"},
    "tawh": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "niin": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "ngam": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawng": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ṭhin": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngar": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    
    # Words frequently tagged as X that have clear tags
    "bei": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "krista": {"wrong_tags": {"X"}, "correct_tag": "NOUN"},
    "ruahmansa": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    "hetah": {"wrong_tags": {"PRON"}, "correct_tag": "ADJ"},
}

# Suffix-based rules for systematic corrections
SUFFIX_CORRECTIONS = {
    # Mizo plural suffix -te on nouns: if tagged ADJ, likely should be NOUN
    # e.g., ṭhalaite, tleirawlte, etc.
}

# ============================================================
# STEP 2: Apply corrections
# ============================================================
print("\n" + "=" * 60)
print("APPLYING CORRECTIONS")
print("=" * 60)

global_fix_count = Counter()
suffix_fix_count = 0
total_tokens_checked = 0

for sent in projected_corpus:
    for tok in sent:
        total_tokens_checked += 1
        token = tok["token"]
        current_tag = tok["final_tag"]
        
        # Clean token for matching (strip trailing punctuation)
        clean = token.lower().rstrip(".,!?;:'\"()-")
        
        # Apply global corrections
        if clean in GLOBAL_CORRECTIONS:
            rule = GLOBAL_CORRECTIONS[clean]
            if current_tag in rule["wrong_tags"]:
                old_tag = current_tag
                tok["final_tag"] = rule["correct_tag"]
                tok["confidence"] = 0.90  # Human-verified correction
                tok["source"] = "human_correction"
                global_fix_count[f"{clean}: {old_tag}→{rule['correct_tag']}"] += 1
        
        # Suffix-based corrections:
        # Words ending in -te that are tagged ADJ but look like plural nouns
        # (Mizo pluralizes with -te suffix on nouns)
        if clean.endswith("te") and len(clean) > 4 and current_tag == "ADJ":
            # Check if the base form (without -te) exists as NOUN in our corrections
            # Be conservative: only fix if the word is long enough and clearly a plural
            # We'll let the human verify in Round 2
            pass
        
        # Words ending in -ah (locative) that are tagged as NOUN 
        # could be ADJ or ADP in context
        # Be conservative here too
        
        # Fix tokens enclosed in quotes that got tagged as PUNCT
        if (token.startswith("'") or token.startswith('"')) and len(clean) > 1:
            if current_tag == "PUNCT":
                # It's a word in quotes, not punctuation
                # Try to find a better tag from lexicon or mark as NOUN (safe default)
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.50
                tok["source"] = "quote_fix"
                global_fix_count["quoted_word: PUNCT→NOUN"] += 1

print(f"\n  Total tokens checked: {total_tokens_checked:,}")
print(f"  Total corrections applied: {sum(global_fix_count.values()):,}")
print(f"\n  Correction breakdown:")
for fix, count in global_fix_count.most_common(30):
    print(f"    {fix:<40} {count:>6,}")

# ============================================================
# STEP 3: Verify impact
# ============================================================
print("\n" + "=" * 60)
print("POST-CORRECTION TAG DISTRIBUTION")
print("=" * 60)

final_dist = Counter()
source_dist = Counter()
for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1
        source_dist[tok["source"]] += 1

total_tokens = sum(final_dist.values())
print(f"\n  Tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

print(f"\n  Source distribution:")
for source, count in source_dist.most_common():
    print(f"    {source:<25} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

# ============================================================
# STEP 4: Save corrected data
# ============================================================
print("\n" + "=" * 60)
print("SAVING CORRECTED DATA")
print("=" * 60)

corrected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_corrected.pkl")
with open(corrected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {corrected_pkl_path}")

# Also update the main file
with open(projected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {projected_pkl_path} (updated)")

# ============================================================
# STEP 5: ROUND 2 SANITY CHECK (15 new sentences)
# ============================================================
print("\n" + "=" * 60)
print("SANITY CHECK - ROUND 2 (15 sentences)")
print("=" * 60)
print()
print("UD Tags: ADJ ADP ADV AUX CCONJ DET INTJ NOUN NUM")
print("         PART PRON PROPN PUNCT SCONJ SYM VERB X")
print()

# Use different seed for new sample, avoid Round 1 sentences
random.seed(202)  # Round 2 seed
round1_indices = {396, 725, 1107, 1427, 2070, 2413, 2549, 4077, 5087, 
                  5426, 5441, 5548, 6048, 6614, 6909, 7106, 7157, 8173, 
                  8483, 8508, 9166, 9765, 10020, 10866, 12556}  # From Round 1

all_indices = [i for i in range(len(projected_corpus)) if i not in round1_indices]
random.shuffle(all_indices)
sample_indices = sorted(all_indices[:15])

for display_num, sent_idx in enumerate(sample_indices, 1):
    sent = projected_corpus[sent_idx]
    en_text = small_en[sent_idx] if sent_idx < len(small_en) else ""
    
    tagged_tokens = [f"{tok['token']}/{tok['final_tag'] or '?'}" for tok in sent]
    
    print(f"\n{'─' * 70}")
    print(f"  S{display_num} (idx={sent_idx})")
    print(f"  EN: {en_text}")
    print(f"  MZ: {' '.join(t['token'] for t in sent)}")
    print(f"  TAG: {' '.join(tagged_tokens)}")
    
    low_conf = [(tok["token"], tok["final_tag"], tok["confidence"], tok["source"]) 
                for tok in sent if tok["confidence"] < 0.70]
    if low_conf:
        print(f"  ⚠️  Low confidence: ", end="")
        print(", ".join([f"'{t}'/{tag} ({conf:.2f},{src})" 
                        for t, tag, conf, src in low_conf]))

print(f"\n{'─' * 70}")
print(f"\n{'=' * 70}")
print("END OF ROUND 2")
print("=" * 70)
print(f"\nSampled sentence indices: {sample_indices}")
print("\nPlease review and report errors in the format:")
print('  S3: "word" should be TAG not CURRENT_TAG')

Loaded 13,155 sentences

APPLYING CORRECTIONS

  Total tokens checked: 157,146
  Total corrections applied: 32,898

  Correction breakdown:
    an: NOUN→PRON                             3,143
    chu: VERB→PART                            3,069
    lo: NOUN→ADJ                              2,892
    kan: NOUN→PRON                            2,666
    ka: NOUN→PRON                             2,528
    chuan: NOUN→CCONJ                         2,134
    leh: VERB→CCONJ                           2,000
    ang: VERB→ADV                             1,750
    hian: ADJ→CCONJ                           1,495
    tih: X→VERB                               1,488
    awm: NOUN→VERB                            1,129
    nge: VERB→ADV                               943
    em: NOUN→ADV                                852
    tawh: NOUN→ADV                              832
    pawh: NOUN→CCONJ                            828
    min: NOUN→PRON                              803
    te: VERB→ADJ            

## Cell 6d: Apply Round 2 Corrections + Round 3 Sanity Check

In [5]:
"""
Cell 6d: Apply Round 2 Corrections + Round 3 Sanity Check
============================================================
"""

import os
import pickle
import random
from collections import Counter

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")

# ============================================================
# Load data
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
print(f"Loaded {len(projected_corpus):,} sentences")

# ============================================================
# GLOBAL CORRECTIONS - Round 2 additions
# ============================================================
# Combined from Round 1 + Round 2 patterns

GLOBAL_CORRECTIONS = {
    # === PRONOUNS (mistagged as NOUN) ===
    "ka": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "kan": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "an": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "min": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "mahni": {"wrong_tags": {"ADJ"}, "correct_tag": "PRON"},
    
    # === CONJUNCTIONS ===
    "chuan": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "leh": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "pawh": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "pawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "hian": {"wrong_tags": {"ADJ"}, "correct_tag": "CCONJ"},
    "niin": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    
    # === PARTICLES ===
    "chu": {"wrong_tags": {"VERB"}, "correct_tag": "PART"},
    
    # === ADVERBS ===
    "ta": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ang": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nge": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "em": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADV"},
    "vek": {"wrong_tags": {"ADJ"}, "correct_tag": "ADV"},
    "tawh": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngam": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawng": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ṭhin": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngar": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zelah": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    "ni": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tur": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawi": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nawn": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "dan": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    
    # === ADJECTIVES ===
    "te": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ten": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "lo": {"wrong_tags": {"NOUN", "ADP"}, "correct_tag": "ADJ"},
    "eng": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADJ"},
    "thei": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hetah": {"wrong_tags": {"PRON"}, "correct_tag": "ADJ"},
    "zing": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "chak": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hun": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "kua": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "dar": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    
    # === VERBS ===
    "tih": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "duh": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "bei": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "awm": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tum": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kam": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "paih": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "ei": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "haw": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "phal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tifai": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},
    "zin": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    
    # === NOUNS ===
    "thawnthu": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "krista": {"wrong_tags": {"X"}, "correct_tag": "NOUN"},
    "kohhran": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "chemtatrawta": {"wrong_tags": {"ADP"}, "correct_tag": "NOUN"},
    "thawhpui": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    
    # === PROPER NOUNS ===
    "isua": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "davida": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "mary": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    
    # === OTHER ===
    "ruahmansa": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
}

# Tokens ending with period/comma that should be PART
# Pattern: single-letter + punctuation like "a.", "a,", "ni."
PART_WITH_PUNCT = {"a.", "a,", "ni."}

# ============================================================
# Apply corrections
# ============================================================
print("\n" + "=" * 60)
print("APPLYING ROUND 2 CORRECTIONS")
print("=" * 60)

fix_count = Counter()
total_tokens = 0

for sent in projected_corpus:
    for tok in sent:
        total_tokens += 1
        token = tok["token"]
        current_tag = tok["final_tag"]
        clean = token.lower().rstrip(".,!?;:'\"()-")
        
        # Check PART with punctuation patterns
        token_lower = token.lower()
        if token_lower in PART_WITH_PUNCT and current_tag != "PART":
            old = current_tag
            tok["final_tag"] = "PART"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r2"
            fix_count[f"{token_lower}: {old}→PART"] += 1
            continue
        
        # Apply global corrections
        if clean in GLOBAL_CORRECTIONS:
            rule = GLOBAL_CORRECTIONS[clean]
            if current_tag in rule["wrong_tags"]:
                old = current_tag
                tok["final_tag"] = rule["correct_tag"]
                tok["confidence"] = 0.90
                tok["source"] = "human_correction_r2"
                fix_count[f"{clean}: {old}→{rule['correct_tag']}"] += 1
        
        # Proper nouns: names ending with 'a' that are tagged X
        # (Common Mizo/biblical names: siama, liani, etc.)
        # Be conservative - only fix if currently X
        if current_tag == "X" and token[0].isupper() and len(clean) > 2:
            tok["final_tag"] = "PROPN"
            tok["confidence"] = 0.70
            tok["source"] = "name_heuristic_r2"
            fix_count[f"capitalized_X→PROPN"] += 1
        
        # Fix quoted words still tagged as PUNCT
        if (token.startswith("'") or token.startswith('"')) and len(clean) > 1:
            if current_tag == "PUNCT":
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.50
                tok["source"] = "quote_fix_r2"
                fix_count["quoted_PUNCT→NOUN"] += 1

print(f"\n  Total tokens checked: {total_tokens:,}")
print(f"  Total corrections: {sum(fix_count.values()):,}")
print(f"\n  Correction breakdown (top 30):")
for fix, count in fix_count.most_common(30):
    print(f"    {fix:<45} {count:>6,}")

if len(fix_count) > 30:
    remaining = sum(c for _, c in list(fix_count.most_common())[30:])
    print(f"    {'... and more':<45} {remaining:>6,}")

# ============================================================
# Post-correction stats
# ============================================================
print("\n" + "=" * 60)
print("POST-CORRECTION STATISTICS")
print("=" * 60)

final_dist = Counter()
source_dist = Counter()
for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1
        source_dist[tok["source"]] += 1

print(f"\n  Tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

human_corrected = sum(1 for sent in projected_corpus for tok in sent 
                      if "human_correction" in tok["source"] or "heuristic_r2" in tok["source"])
print(f"\n  Human-corrected tokens: {human_corrected:,} ({100*human_corrected/total_tokens:.1f}%)")

# ============================================================
# Save corrected data
# ============================================================
print("\n  Saving...")
with open(projected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {projected_pkl_path} (updated)")

corrected_r2_path = os.path.join(OUTPUT_DIR, "projected_corpus_corrected_r2.pkl")
with open(corrected_r2_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {corrected_r2_path}")

# ============================================================
# ROUND 3 SANITY CHECK (15 new sentences)
# ============================================================
print("\n" + "=" * 70)
print("SANITY CHECK - ROUND 3 (15 sentences)")
print("=" * 70)
print()
print("UD Tags: ADJ ADP ADV AUX CCONJ DET INTJ NOUN NUM")
print("         PART PRON PROPN PUNCT SCONJ SYM VERB X")
print()

random.seed(303)  # Round 3 seed
# Exclude previous round indices
prev_indices = {396, 725, 1107, 1427, 2070, 2413, 2549, 4077, 5087, 
                5426, 5441, 5548, 6048, 6614, 6909, 7106, 7157, 8173, 
                8483, 8508, 9166, 9765, 10020, 10866, 12556}

all_indices = [i for i in range(len(projected_corpus)) if i not in prev_indices]
random.shuffle(all_indices)
sample_indices = sorted(all_indices[:15])

for display_num, sent_idx in enumerate(sample_indices, 1):
    sent = projected_corpus[sent_idx]
    en_text = small_en[sent_idx] if sent_idx < len(small_en) else ""
    
    tagged_tokens = [f"{tok['token']}/{tok['final_tag'] or '?'}" for tok in sent]
    
    print(f"\n{'─' * 70}")
    print(f"  S{display_num} (idx={sent_idx})")
    print(f"  EN: {en_text}")
    print(f"  MZ: {' '.join(t['token'] for t in sent)}")
    print(f"  TAG: {' '.join(tagged_tokens)}")
    
    low_conf = [(tok["token"], tok["final_tag"], tok["confidence"], tok["source"]) 
                for tok in sent if tok["confidence"] < 0.70]
    if low_conf:
        print(f"  ⚠️  Low confidence: ", end="")
        print(", ".join([f"'{t}'/{tag} ({conf:.2f},{src})" 
                        for t, tag, conf, src in low_conf]))

print(f"\n{'─' * 70}")
print(f"\n{'=' * 70}")
print("END OF ROUND 3")
print("=" * 70)
print(f"\nSampled sentence indices: {sample_indices}")
print("\nPlease review and report errors.")

Loaded 13,155 sentences

APPLYING ROUND 2 CORRECTIONS

  Total tokens checked: 157,146
  Total corrections: 11,757

  Correction breakdown (top 30):
    ni.: VERB→PART                                 2,344
    ni: VERB→ADV                                   2,117
    thei: NOUN→ADJ                                 1,037
    tur: VERB→ADV                                    898
    a.: NOUN→PART                                    890
    eng: NOUN→ADJ                                    638
    kal: NOUN→VERB                                   586
    hun: NOUN→ADJ                                    362
    a,: NOUN→PART                                    352
    dan: NOUN→ADV                                    258
    tum: NOUN→VERB                                   256
    a.: PRON→PART                                    228
    ni.: ADV→PART                                    207
    pawhin: NOUN→CCONJ                               199
    kohhran: ADJ→NOUN                                

## Cell 6e: Apply Round 3 Corrections + Round 4 Sanity Check

In [6]:
"""
Cell 6e: Apply Round 3 Corrections + Round 4 Sanity Check
============================================================
"""

import os
import pickle
import random
from collections import Counter

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")

# ============================================================
# Load data
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
print(f"Loaded {len(projected_corpus):,} sentences")

# ============================================================
# CUMULATIVE GLOBAL CORRECTIONS (R1 + R2 + R3)
# ============================================================

GLOBAL_CORRECTIONS = {
    # === PRONOUNS ===
    "ka": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "kan": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "an": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "PRON"},
    "min": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "mahni": {"wrong_tags": {"ADJ"}, "correct_tag": "PRON"},
    "i": {"wrong_tags": {"PART", "NOUN"}, "correct_tag": "PRON"},
    "in": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    
    # === CONJUNCTIONS ===
    "chuan": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "leh": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "pawh": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "pawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "hian": {"wrong_tags": {"ADJ"}, "correct_tag": "CCONJ"},
    "niin": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "mahse": {"wrong_tags": {"ADV"}, "correct_tag": "CCONJ"},  # R3 new
    
    # === PARTICLES ===
    "chu": {"wrong_tags": {"VERB"}, "correct_tag": "PART"},
    "ve": {"wrong_tags": {"NOUN"}, "correct_tag": "PART"},  # R3 new
    
    # === ADVERBS ===
    "ta": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ang": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nge": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "em": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADV"},
    "vek": {"wrong_tags": {"ADJ"}, "correct_tag": "ADV"},
    "tawh": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngam": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawng": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ṭhin": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngar": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zelah": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    "ni": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tur": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawi": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nawn": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "dan": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngawih": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},  # R3 new
    "mai": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},     # R3 new
    "hlim": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},    # R3 new
    "lawm": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},    # R3 new
    "dawn": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},    # R3 new
    "ruahmansa": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    
    # === ADJECTIVES ===
    "te": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ten": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "lo": {"wrong_tags": {"NOUN", "ADP"}, "correct_tag": "ADJ"},
    "eng": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADJ"},
    "thei": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hetah": {"wrong_tags": {"PRON"}, "correct_tag": "ADJ"},
    "zing": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "chak": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hun": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "kua": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "dar": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "chawhnu": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},  # R3 new
    "thum": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},     # R3 new
    "tun": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},      # R3 new
    "ṭha": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},      # R3 new
    "ṭhat": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},     # R3 new
    "heng": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},     # R3 new
    "hrang": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},    # R3 new
    "bawk": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},     # R3 new
    "taka": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},     # R3 new
    "fel": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    
    # === VERBS ===
    "tih": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "duh": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "bei": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "awm": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tum": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kam": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "paih": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "ei": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "haw": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "phal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tifai": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},
    "zin": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "pui": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},       # R3 new
    "riang": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},     # R3 new
    "chhiar": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},    # R3 new
    "thlen": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},     # R3 new
    "chhuah": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},    # R3 new
    "phun": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},      # R3 new
    "chhang": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},    # R3 new
    
    # === NOUNS ===
    "thawnthu": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "krista": {"wrong_tags": {"X"}, "correct_tag": "NOUN"},
    "kohhran": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "chemtatrawta": {"wrong_tags": {"ADP"}, "correct_tag": "NOUN"},
    "thawhpui": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "lehkhabu": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},   # R3 new
    
    # === PROPER NOUNS ===
    "isua": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "davida": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "mary": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "samuelan": {"wrong_tags": {"DET"}, "correct_tag": "PROPN"},   # R3 new
    "sen": {"wrong_tags": {"NOUN"}, "correct_tag": "PROPN"},       # R3 new (context-dep but often a name)
}

# Tokens that are PART when followed by punctuation
PART_WITH_PUNCT = {"a.", "a,", "ni."}

# ============================================================
# Apply corrections
# ============================================================
print("\n" + "=" * 60)
print("APPLYING ROUND 3 CORRECTIONS")
print("=" * 60)

fix_count = Counter()
total_tokens = 0

for sent in projected_corpus:
    for tok in sent:
        total_tokens += 1
        token = tok["token"]
        current_tag = tok["final_tag"]
        clean = token.lower().rstrip(".,!?;:'\"()-")
        token_lower = token.lower()
        
        # PART with punctuation
        if token_lower in PART_WITH_PUNCT and current_tag != "PART":
            old = current_tag
            tok["final_tag"] = "PART"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r3"
            fix_count[f"{token_lower}: {old}→PART"] += 1
            continue
        
        # Global corrections
        if clean in GLOBAL_CORRECTIONS:
            rule = GLOBAL_CORRECTIONS[clean]
            if current_tag in rule["wrong_tags"]:
                old = current_tag
                tok["final_tag"] = rule["correct_tag"]
                tok["confidence"] = 0.90
                tok["source"] = "human_correction_r3"
                fix_count[f"{clean}: {old}→{rule['correct_tag']}"] += 1
        
        # Capitalized X → PROPN
        if current_tag == "X" and token[0].isupper() and len(clean) > 2:
            tok["final_tag"] = "PROPN"
            tok["confidence"] = 0.70
            tok["source"] = "name_heuristic_r3"
            fix_count["capitalized_X→PROPN"] += 1
        
        # Quoted words tagged PUNCT
        if (token.startswith("'") or token.startswith('"')) and len(clean) > 1:
            if current_tag == "PUNCT":
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.50
                tok["source"] = "quote_fix_r3"
                fix_count["quoted_PUNCT→NOUN"] += 1
        
        # Mizo plural -te suffix: words ending in "te" tagged ADJ → likely NOUN
        # e.g., mizote, ṭhalaite, tleirawlte
        if clean.endswith("te") and len(clean) > 4 and current_tag == "ADJ":
            # Check it's not a known ADJ
            known_adj_te = {"chite"}  # exceptions
            if clean not in known_adj_te:
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.80
                tok["source"] = "plural_te_fix_r3"
                fix_count["plural_-te ADJ→NOUN"] += 1

print(f"\n  Total tokens checked: {total_tokens:,}")
print(f"  Total corrections: {sum(fix_count.values()):,}")
print(f"\n  Correction breakdown (top 30):")
for fix, count in fix_count.most_common(30):
    print(f"    {fix:<45} {count:>6,}")
if len(fix_count) > 30:
    remaining = sum(c for _, c in list(fix_count.most_common())[30:])
    print(f"    {'... and more':<45} {remaining:>6,}")

# ============================================================
# Post-correction stats
# ============================================================
print("\n" + "=" * 60)
print("POST-CORRECTION STATISTICS")
print("=" * 60)

final_dist = Counter()
for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1

print(f"\n  Tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

human_corrected = sum(1 for sent in projected_corpus for tok in sent 
                      if "human_correction" in tok["source"] or "heuristic_r" in tok["source"]
                      or "plural_te" in tok["source"] or "name_heuristic" in tok["source"])
print(f"\n  Human-corrected tokens (cumulative): {human_corrected:,} ({100*human_corrected/total_tokens:.1f}%)")

# X tag count
x_count = final_dist.get("X", 0)
print(f"  Remaining X tags: {x_count:,} ({100*x_count/total_tokens:.1f}%)")

# ============================================================
# Save
# ============================================================
print("\n  Saving...")
with open(projected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {projected_pkl_path} (updated)")

# ============================================================
# ROUND 4 SANITY CHECK (15 new sentences)
# ============================================================
print("\n" + "=" * 70)
print("SANITY CHECK - ROUND 4 (15 sentences)")
print("=" * 70)
print()
print("UD Tags: ADJ ADP ADV AUX CCONJ DET INTJ NOUN NUM")
print("         PART PRON PROPN PUNCT SCONJ SYM VERB X")
print()

random.seed(404)
# Exclude all previous round indices
prev_indices = {396, 725, 1107, 1427, 2070, 2413, 2549, 4077, 5087, 
                5426, 5441, 5548, 6048, 6614, 6909, 7106, 7157, 8173, 
                8483, 8508, 9166, 9765, 10020, 10866, 12556}

all_indices = [i for i in range(len(projected_corpus)) if i not in prev_indices]
random.shuffle(all_indices)
sample_indices = sorted(all_indices[:15])

for display_num, sent_idx in enumerate(sample_indices, 1):
    sent = projected_corpus[sent_idx]
    en_text = small_en[sent_idx] if sent_idx < len(small_en) else ""
    
    tagged_tokens = [f"{tok['token']}/{tok['final_tag'] or '?'}" for tok in sent]
    
    print(f"\n{'─' * 70}")
    print(f"  S{display_num} (idx={sent_idx})")
    print(f"  EN: {en_text}")
    print(f"  MZ: {' '.join(t['token'] for t in sent)}")
    print(f"  TAG: {' '.join(tagged_tokens)}")
    
    low_conf = [(tok["token"], tok["final_tag"], tok["confidence"], tok["source"]) 
                for tok in sent if tok["confidence"] < 0.70]
    if low_conf:
        print(f"  ⚠️  Low confidence: ", end="")
        print(", ".join([f"'{t}'/{tag} ({conf:.2f},{src})" 
                        for t, tag, conf, src in low_conf]))

print(f"\n{'─' * 70}")
print(f"\n{'=' * 70}")
print("END OF ROUND 4")
print("=" * 70)
print(f"\nSampled sentence indices: {sample_indices}")
print("\nPlease review and report errors.")

Loaded 13,155 sentences

APPLYING ROUND 3 CORRECTIONS

  Total tokens checked: 157,146
  Total corrections: 7,261

  Correction breakdown (top 30):
    i: NOUN→PRON                                   1,531
    mai: VERB→ADV                                  1,013
    plural_-te ADJ→NOUN                              845
    ṭha: VERB→ADJ                                    665
    ve: NOUN→PART                                    554
    dawn: NOUN→ADV                                   444
    bawk: NOUN→ADJ                                   382
    hrang: NOUN→ADJ                                  274
    chhuah: NOUN→VERB                                195
    mahse: ADV→CCONJ                                 168
    lawm: NOUN→ADV                                   162
    thlen: NOUN→VERB                                 159
    pui: NOUN→VERB                                    94
    hlim: NOUN→ADV                                    93
    chhiar: NOUN→VERB                                 

## Cell 6f: Apply Round 4 Corrections + Round 5 Sanity Check (Final Round)

In [7]:
"""
Cell 6f: Apply Round 4 Corrections + Round 5 Sanity Check (Final Round)
=========================================================================
"""

import os
import pickle
import random
from collections import Counter

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")

# ============================================================
# Load data
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
print(f"Loaded {len(projected_corpus):,} sentences")

# ============================================================
# CUMULATIVE GLOBAL CORRECTIONS (R1 + R2 + R3 + R4)
# ============================================================

GLOBAL_CORRECTIONS = {
    # === PRONOUNS ===
    "ka": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "kan": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "an": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "PRON"},
    "min": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "mahni": {"wrong_tags": {"ADJ"}, "correct_tag": "PRON"},
    "i": {"wrong_tags": {"PART", "NOUN"}, "correct_tag": "PRON"},
    "in": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    
    # === CONJUNCTIONS ===
    "chuan": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "CCONJ"},  # R4: added VERB
    "leh": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "pawh": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "pawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "hian": {"wrong_tags": {"ADJ"}, "correct_tag": "CCONJ"},
    "niin": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "mahse": {"wrong_tags": {"ADV"}, "correct_tag": "CCONJ"},
    
    # === PARTICLES ===
    "chu": {"wrong_tags": {"VERB"}, "correct_tag": "PART"},
    "ve": {"wrong_tags": {"NOUN"}, "correct_tag": "PART"},
    
    # === ADVERBS ===
    "ta": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ang": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nge": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "em": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADV"},
    "vek": {"wrong_tags": {"ADJ"}, "correct_tag": "ADV"},
    "tawh": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngam": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawng": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ṭhin": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngar": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zelah": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    "ni": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tur": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawi": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nawn": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "dan": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngawih": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "mai": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "hlim": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "lawm": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "dawn": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ruahmansa": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    "mah": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},       # R4 new
    "tûnlai": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},    # R4 new
    "theih": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},     # R4 new (was ADJ in R2, ADV in R4)
    
    # === ADJECTIVES ===
    "te": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ten": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "lo": {"wrong_tags": {"NOUN", "ADP"}, "correct_tag": "ADJ"},
    "eng": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADJ"},
    "thei": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hetah": {"wrong_tags": {"PRON"}, "correct_tag": "ADJ"},
    "zing": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "chak": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hun": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "kua": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "dar": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "chawhnu": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "thum": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tun": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ṭha": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ṭhat": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "heng": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hrang": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "bawk": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "taka": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "fel": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tam": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},       # R4 new
    "tak": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},       # R4 new
    "ber": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},       # R4 new
    "ngai": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},      # R4 new
    "lâr": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},       # R4 new
    "vang": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},      # R4 new
    
    # === VERBS ===
    "tih": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "duh": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "bei": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "awm": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tum": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kam": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "paih": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "ei": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "haw": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "phal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tifai": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},
    "zin": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "pui": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "riang": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "chhiar": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "thlen": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "chhuah": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "phun": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "chhang": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "zak": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},       # R4 new
    "dil": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},       # R4 new
    "pên": {"wrong_tags": {"X"}, "correct_tag": "VERB"},          # R4 new
    "tlânsan": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},    # R4 new
    "chang": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},      # R4 new
    "neiin": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},      # R4 new
    "chhawr": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},    # R4 new
    "hmelṭhatpui": {"wrong_tags": {"X"}, "correct_tag": "VERB"},  # R4 new
    
    # === NOUNS ===
    "thawnthu": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "krista": {"wrong_tags": {"X"}, "correct_tag": "NOUN"},
    "kohhran": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "chemtatrawta": {"wrong_tags": {"ADP"}, "correct_tag": "NOUN"},
    "thawhpui": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "lehkhabu": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "vun": {"wrong_tags": {"AUX"}, "correct_tag": "NOUN"},        # R4 new
    
    # === PROPER NOUNS ===
    "isua": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "davida": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "mary": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "samuelan": {"wrong_tags": {"DET"}, "correct_tag": "PROPN"},
    "thlenga": {"wrong_tags": {"ADJ"}, "correct_tag": "PROPN"},   # R4 new
}

# Suffix-based corrections for words ending in -ah (locative marker → ADV)
# e.g., hmalaknaah → ADV, not X
SUFFIX_AH_TO_ADV = True  # Apply -ah suffix rule for X-tagged words

# Tokens that are PART when followed by punctuation
PART_WITH_PUNCT = {"a.", "a,", "ni."}

# ============================================================
# Apply corrections
# ============================================================
print("\n" + "=" * 60)
print("APPLYING ROUND 4 CORRECTIONS")
print("=" * 60)

fix_count = Counter()
total_tokens = 0

for sent in projected_corpus:
    for tok in sent:
        total_tokens += 1
        token = tok["token"]
        current_tag = tok["final_tag"]
        clean = token.lower().rstrip(".,!?;:'\"()-")
        token_lower = token.lower()
        
        # PART with punctuation
        if token_lower in PART_WITH_PUNCT and current_tag != "PART":
            old = current_tag
            tok["final_tag"] = "PART"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r4"
            fix_count[f"{token_lower}: {old}→PART"] += 1
            continue
        
        # "ngai." specifically → ADV (with period)
        if token_lower == "ngai." and current_tag != "ADV":
            old = current_tag
            tok["final_tag"] = "ADV"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r4"
            fix_count[f"ngai.: {old}→ADV"] += 1
            continue
        
        # "hrilh." → VERB
        if clean == "hrilh" and current_tag == "NOUN":
            tok["final_tag"] = "VERB"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r4"
            fix_count["hrilh: NOUN→VERB"] += 1
            continue
        
        # Global corrections
        if clean in GLOBAL_CORRECTIONS:
            rule = GLOBAL_CORRECTIONS[clean]
            if current_tag in rule["wrong_tags"]:
                old = current_tag
                tok["final_tag"] = rule["correct_tag"]
                tok["confidence"] = 0.90
                tok["source"] = "human_correction_r4"
                fix_count[f"{clean}: {old}→{rule['correct_tag']}"] += 1
        
        # Suffix -ah on X-tagged words → ADV (locative)
        if SUFFIX_AH_TO_ADV and current_tag == "X" and clean.endswith("ah") and len(clean) > 3:
            tok["final_tag"] = "ADV"
            tok["confidence"] = 0.75
            tok["source"] = "suffix_ah_r4"
            fix_count["suffix -ah X→ADV"] += 1
        
        # Capitalized X → PROPN
        if current_tag == "X" and len(token) > 2 and token[0].isupper():
            tok["final_tag"] = "PROPN"
            tok["confidence"] = 0.70
            tok["source"] = "name_heuristic_r4"
            fix_count["capitalized_X→PROPN"] += 1
        
        # Quoted words tagged PUNCT
        if (token.startswith("'") or token.startswith('"')) and len(clean) > 1:
            if current_tag == "PUNCT":
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.50
                tok["source"] = "quote_fix_r4"
                fix_count["quoted_PUNCT→NOUN"] += 1
        
        # Plural -te suffix: ADJ → NOUN
        if clean.endswith("te") and len(clean) > 4 and current_tag == "ADJ":
            known_adj_te = {"chite", "zawngte"}
            if clean not in known_adj_te:
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.80
                tok["source"] = "plural_te_fix_r4"
                fix_count["plural -te ADJ→NOUN"] += 1

print(f"\n  Total tokens checked: {total_tokens:,}")
print(f"  Total corrections: {sum(fix_count.values()):,}")
print(f"\n  Correction breakdown (top 30):")
for fix, count in fix_count.most_common(30):
    print(f"    {fix:<45} {count:>6,}")
if len(fix_count) > 30:
    remaining = sum(c for _, c in list(fix_count.most_common())[30:])
    print(f"    {'... and more':<45} {remaining:>6,}")

# ============================================================
# Post-correction stats
# ============================================================
print("\n" + "=" * 60)
print("POST-CORRECTION STATISTICS")
print("=" * 60)

final_dist = Counter()
for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1

print(f"\n  Tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

x_count = final_dist.get("X", 0)
print(f"\n  Remaining X tags: {x_count:,} ({100*x_count/total_tokens:.2f}%)")

# ============================================================
# Save
# ============================================================
print("\n  Saving...")
with open(projected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {projected_pkl_path} (updated)")

# ============================================================
# ROUND 5 SANITY CHECK - FINAL ROUND (15 sentences)
# ============================================================
print("\n" + "=" * 70)
print("SANITY CHECK - ROUND 5 / FINAL ROUND (15 sentences)")
print("=" * 70)
print()
print("UD Tags: ADJ ADP ADV AUX CCONJ DET INTJ NOUN NUM")
print("         PART PRON PROPN PUNCT SCONJ SYM VERB X")
print()

random.seed(505)
prev_indices = {396, 725, 1107, 1427, 2070, 2413, 2549, 4077, 5087, 
                5426, 5441, 5548, 6048, 6614, 6909, 7106, 7157, 8173, 
                8483, 8508, 9166, 9765, 10020, 10866, 12556}

all_indices = [i for i in range(len(projected_corpus)) if i not in prev_indices]
random.shuffle(all_indices)
sample_indices = sorted(all_indices[:15])

for display_num, sent_idx in enumerate(sample_indices, 1):
    sent = projected_corpus[sent_idx]
    en_text = small_en[sent_idx] if sent_idx < len(small_en) else ""
    
    tagged_tokens = [f"{tok['token']}/{tok['final_tag'] or '?'}" for tok in sent]
    
    print(f"\n{'─' * 70}")
    print(f"  S{display_num} (idx={sent_idx})")
    print(f"  EN: {en_text}")
    print(f"  MZ: {' '.join(t['token'] for t in sent)}")
    print(f"  TAG: {' '.join(tagged_tokens)}")
    
    low_conf = [(tok["token"], tok["final_tag"], tok["confidence"], tok["source"]) 
                for tok in sent if tok["confidence"] < 0.70]
    if low_conf:
        print(f"  ⚠️  Low confidence: ", end="")
        print(", ".join([f"'{t}'/{tag} ({conf:.2f},{src})" 
                        for t, tag, conf, src in low_conf]))

print(f"\n{'─' * 70}")
print(f"\n{'=' * 70}")
print("END OF ROUND 5 (FINAL)")
print("=" * 70)
print(f"\nSampled sentence indices: {sample_indices}")
print("\nPlease review. If error rate is low (<5 errors per 15 sentences),")
print("we can proceed to re-export final training data and then")
print("scale to the large corpus.")

Loaded 13,155 sentences

APPLYING ROUND 4 CORRECTIONS

  Total tokens checked: 157,146
  Total corrections: 3,067

  Correction breakdown (top 30):
    theih: NOUN→ADV                                  526
    ngai: VERB→ADJ                                   489
    tam: NOUN→ADJ                                    487
    ber: VERB→ADJ                                    323
    chuan: VERB→CCONJ                                282
    suffix -ah X→ADV                                 257
    hrilh: NOUN→VERB                                 251
    mah: VERB→ADV                                    106
    vang: NOUN→ADJ                                    86
    ngai.: VERB→ADV                                   78
    dil: NOUN→VERB                                    40
    thlenga: ADJ→PROPN                                39
    ngai.: NOUN→ADV                                   21
    tûnlai: NOUN→ADV                                  16
    neiin: ADJ→VERB                                   

## Cell 6g: Apply Round 5 Corrections + X-Tag Review + Re-export Training Data

In [8]:
"""
Cell 6g: Apply Round 5 Corrections + X-Tag Review + Re-export Training Data
==============================================================================
Final correction round, then re-export clean train/dev/test splits.
"""

import os
import pickle
import random
import json
import statistics
from collections import Counter, defaultdict

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")

# ============================================================
# Load data
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
print(f"Loaded {len(projected_corpus):,} sentences")

# ============================================================
# CUMULATIVE GLOBAL CORRECTIONS (R1-R5 ALL)
# ============================================================

GLOBAL_CORRECTIONS = {
    # === PRONOUNS ===
    "ka": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "kan": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "an": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "PRON"},
    "min": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    "mahni": {"wrong_tags": {"ADJ"}, "correct_tag": "PRON"},
    "i": {"wrong_tags": {"PART", "NOUN"}, "correct_tag": "PRON"},
    "in": {"wrong_tags": {"NOUN"}, "correct_tag": "PRON"},
    
    # === CONJUNCTIONS ===
    "chuan": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "CCONJ"},
    "leh": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "pawh": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "pawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "CCONJ"},
    "hian": {"wrong_tags": {"ADJ"}, "correct_tag": "CCONJ"},
    "niin": {"wrong_tags": {"VERB"}, "correct_tag": "CCONJ"},
    "mahse": {"wrong_tags": {"ADV"}, "correct_tag": "CCONJ"},
    
    # === PARTICLES ===
    "chu": {"wrong_tags": {"VERB"}, "correct_tag": "PART"},
    "ve": {"wrong_tags": {"NOUN"}, "correct_tag": "PART"},
    
    # === ADVERBS ===
    "ta": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ang": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nge": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "em": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADV"},
    "vek": {"wrong_tags": {"ADJ"}, "correct_tag": "ADV"},
    "tawh": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngam": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawng": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "ṭhin": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tawhin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngar": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zelah": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    "ni": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tur": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "zawi": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "nawn": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "dan": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ngawih": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "mai": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "hlim": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "lawm": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "dawn": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "ruahmansa": {"wrong_tags": {"X"}, "correct_tag": "ADV"},
    "mah": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},
    "tûnlai": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "theih": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},
    "zel": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},       # R5 new
    "rei": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},       # R5 new
    "takte": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},     # R5 new
    "han": {"wrong_tags": {"NOUN"}, "correct_tag": "ADV"},       # R5 new
    "kawthalo": {"wrong_tags": {"VERB"}, "correct_tag": "ADV"},  # R5 new
    
    # === ADJECTIVES ===
    "te": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ten": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "lo": {"wrong_tags": {"NOUN", "ADP"}, "correct_tag": "ADJ"},
    "eng": {"wrong_tags": {"NOUN", "VERB"}, "correct_tag": "ADJ"},
    "thei": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hetah": {"wrong_tags": {"PRON"}, "correct_tag": "ADJ"},
    "zing": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tin": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "chak": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hun": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "kua": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "dar": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "chawhnu": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "thum": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tun": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ṭha": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ṭhat": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "heng": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hrang": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "bawk": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "taka": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "fel": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "tam": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "tak": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "ber": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "ngai": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},
    "lâr": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "vang": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},
    "hma": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},       # R5 new
    "lam": {"wrong_tags": {"NOUN"}, "correct_tag": "ADJ"},       # R5 new
    "nuam": {"wrong_tags": {"VERB"}, "correct_tag": "ADJ"},      # R5 new
    
    # === VERBS ===
    "tih": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "duh": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "bei": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "awm": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tum": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kam": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "kal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "paih": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "ei": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "haw": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "phal": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "tifai": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},
    "zin": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "pui": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "riang": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "chhiar": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "thlen": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "chhuah": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "phun": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "chhang": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "zak": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "dil": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "pên": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "tlânsan": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},
    "chang": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},
    "neiin": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},
    "chhawr": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},
    "hmelṭhatpui": {"wrong_tags": {"X"}, "correct_tag": "VERB"},
    "zai": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},       # R5 new
    "hman": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},      # R5 new
    "thi": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},       # R5 new
    "lang": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},      # R5 new
    "pek": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},       # R5 new
    "sawiho": {"wrong_tags": {"ADJ"}, "correct_tag": "VERB"},     # R5 new
    "pan": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},       # R5 new
    "ngen": {"wrong_tags": {"NOUN"}, "correct_tag": "VERB"},      # R5 new
    "tihhlawhtlin": {"wrong_tags": {"ADV"}, "correct_tag": "VERB"},# R5 new
    "dahlêt": {"wrong_tags": {"X"}, "correct_tag": "VERB"},       # R5 new
    
    # === NOUNS ===
    "thawnthu": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "krista": {"wrong_tags": {"X"}, "correct_tag": "NOUN"},
    "kohhran": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "chemtatrawta": {"wrong_tags": {"ADP"}, "correct_tag": "NOUN"},
    "thawhpui": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "lehkhabu": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},
    "vun": {"wrong_tags": {"AUX"}, "correct_tag": "NOUN"},
    "pathian": {"wrong_tags": {"PUNCT"}, "correct_tag": "NOUN"},   # R5 new
    "lalpa": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"},      # R5 new
    "ngaihsamna": {"wrong_tags": {"ADJ"}, "correct_tag": "NOUN"}, # R5 new
    
    # === PROPER NOUNS ===
    "isua": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "davida": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "mary": {"wrong_tags": {"X", "NOUN"}, "correct_tag": "PROPN"},
    "samuelan": {"wrong_tags": {"DET"}, "correct_tag": "PROPN"},
    "thlenga": {"wrong_tags": {"ADJ"}, "correct_tag": "PROPN"},
    "tom": {"wrong_tags": {"VERB"}, "correct_tag": "PROPN"},      # R5 new
}

PART_WITH_PUNCT = {"a.", "a,", "ni."}

# ============================================================
# Apply corrections
# ============================================================
print("\n" + "=" * 60)
print("APPLYING ROUND 5 (FINAL) CORRECTIONS")
print("=" * 60)

fix_count = Counter()
total_tokens = 0

for sent in projected_corpus:
    for tok in sent:
        total_tokens += 1
        token = tok["token"]
        current_tag = tok["final_tag"]
        clean = token.lower().rstrip(".,!?;:'\"()-")
        token_lower = token.lower()
        
        # PART with punctuation
        if token_lower in PART_WITH_PUNCT and current_tag != "PART":
            old = current_tag
            tok["final_tag"] = "PART"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r5"
            fix_count[f"{token_lower}: {old}→PART"] += 1
            continue
        
        # ngai. → ADV
        if token_lower == "ngai." and current_tag != "ADV":
            tok["final_tag"] = "ADV"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r5"
            fix_count[f"ngai.: {current_tag}→ADV"] += 1
            continue
        
        # hrilh → VERB
        if clean == "hrilh" and current_tag == "NOUN":
            tok["final_tag"] = "VERB"
            tok["confidence"] = 0.90
            tok["source"] = "human_correction_r5"
            fix_count["hrilh: NOUN→VERB"] += 1
            continue
        
        # Global corrections
        if clean in GLOBAL_CORRECTIONS:
            rule = GLOBAL_CORRECTIONS[clean]
            if current_tag in rule["wrong_tags"]:
                old = current_tag
                tok["final_tag"] = rule["correct_tag"]
                tok["confidence"] = 0.90
                tok["source"] = "human_correction_r5"
                fix_count[f"{clean}: {old}→{rule['correct_tag']}"] += 1
        
        # Suffix -ah on X-tagged → ADV
        if tok["final_tag"] == "X" and clean.endswith("ah") and len(clean) > 3:
            tok["final_tag"] = "ADV"
            tok["confidence"] = 0.75
            tok["source"] = "suffix_ah_r5"
            fix_count["suffix -ah X→ADV"] += 1
        
        # Capitalized X → PROPN
        if tok["final_tag"] == "X" and len(token) > 2 and token[0].isupper():
            tok["final_tag"] = "PROPN"
            tok["confidence"] = 0.70
            tok["source"] = "name_heuristic_r5"
            fix_count["capitalized_X→PROPN"] += 1
        
        # Quoted PUNCT fix
        if (token.startswith("'") or token.startswith('"')) and len(clean) > 1:
            if tok["final_tag"] == "PUNCT":
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.50
                tok["source"] = "quote_fix_r5"
                fix_count["quoted_PUNCT→NOUN"] += 1
        
        # Plural -te ADJ→NOUN
        if clean.endswith("te") and len(clean) > 4 and tok["final_tag"] == "ADJ":
            known_adj_te = {"chite", "zawngte"}
            if clean not in known_adj_te:
                tok["final_tag"] = "NOUN"
                tok["confidence"] = 0.80
                tok["source"] = "plural_te_fix_r5"
                fix_count["plural -te ADJ→NOUN"] += 1

total_fixes = sum(fix_count.values())
print(f"\n  Total tokens: {total_tokens:,}")
print(f"  Total corrections: {total_fixes:,}")
print(f"\n  Top corrections:")
for fix, count in fix_count.most_common(25):
    print(f"    {fix:<45} {count:>6,}")

# ============================================================
# X-TAG REVIEW: List all unique X-tagged tokens
# ============================================================
print("\n" + "=" * 60)
print("X-TAG REVIEW")
print("=" * 60)

x_tokens = Counter()
for sent in projected_corpus:
    for tok in sent:
        if tok["final_tag"] == "X":
            clean = tok["token"].lower().rstrip(".,!?;:'\"()-")
            x_tokens[clean] += 1

print(f"\n  Total X-tagged tokens: {sum(x_tokens.values()):,}")
print(f"  Unique X-tagged forms: {len(x_tokens):,}")
print(f"\n  All X-tagged tokens (frequency ≥ 2):")
for token, count in x_tokens.most_common():
    if count >= 2:
        print(f"    {token:<30} {count:>5}")

print(f"\n  X-tagged tokens appearing only once: {sum(1 for t, c in x_tokens.items() if c == 1)}")

# ============================================================
# POST-CORRECTION FINAL STATISTICS
# ============================================================
print("\n" + "=" * 60)
print("FINAL CORPUS STATISTICS")
print("=" * 60)

final_dist = Counter()
source_dist = Counter()
conf_list = []
for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1
        source_dist[tok["source"]] += 1
        conf_list.append(tok["confidence"])

print(f"\n  Tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

print(f"\n  Source distribution:")
for source, count in source_dist.most_common():
    print(f"    {source:<30} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

print(f"\n  Confidence: mean={statistics.mean(conf_list):.3f}, "
      f"median={statistics.median(conf_list):.3f}")

# ============================================================
# RE-EXPORT TRAINING DATA
# ============================================================
print("\n" + "=" * 60)
print("RE-EXPORTING CORRECTED TRAINING DATA")
print("=" * 60)

# Rebuild tiers
tier1, tier2, tier3 = [], [], []
small_mz = read_lines(os.path.join(DATA_DIR, "small.mz"))

for sent_idx, sent in enumerate(projected_corpus):
    token_tags = [(tok["token"], tok["final_tag"]) for tok in sent]
    min_conf = min(tok["confidence"] for tok in sent)
    mean_conf = statistics.mean(tok["confidence"] for tok in sent)
    
    entry = {
        "idx": sent_idx,
        "tokens": token_tags,
        "min_conf": min_conf,
        "mean_conf": mean_conf,
        "en_text": small_en[sent_idx] if sent_idx < len(small_en) else "",
    }
    
    tier3.append(entry)
    if min_conf >= 0.50:
        tier2.append(entry)
    if min_conf >= 0.70:
        tier1.append(entry)

print(f"  Tier 1 (high, min_conf>=0.70): {len(tier1):>8,}")
print(f"  Tier 2 (med,  min_conf>=0.50): {len(tier2):>8,}")
print(f"  Tier 3 (all):                  {len(tier3):>8,}")

# Train/Dev/Test splits
random.seed(42)
tier1_shuffled = tier1.copy()
random.shuffle(tier1_shuffled)

n_dev = max(500, len(tier1_shuffled) // 10)
n_test = max(500, len(tier1_shuffled) // 10)

if n_dev + n_test > len(tier1_shuffled):
    n_dev = len(tier1_shuffled) // 3
    n_test = len(tier1_shuffled) // 3

test_set = tier1_shuffled[:n_test]
dev_set = tier1_shuffled[n_test:n_test + n_dev]

dev_test_indices = set(s["idx"] for s in dev_set + test_set)
train_set = [s for s in tier2 if s["idx"] not in dev_test_indices]

print(f"\n  Train: {len(train_set):>8,} sentences")
print(f"  Dev:   {len(dev_set):>8,} sentences")
print(f"  Test:  {len(test_set):>8,} sentences")

train_tokens = sum(len(s["tokens"]) for s in train_set)
dev_tokens = sum(len(s["tokens"]) for s in dev_set)
test_tokens = sum(len(s["tokens"]) for s in test_set)
print(f"  Train tokens: {train_tokens:>10,}")
print(f"  Dev tokens:   {dev_tokens:>10,}")
print(f"  Test tokens:  {test_tokens:>10,}")

# Export CoNLL files
def export_conll(sentences, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for sent in sentences:
            for token, tag in sent["tokens"]:
                f.write(f"{token}\t{tag}\n")
            f.write("\n")
    return os.path.getsize(filepath) / (1024 * 1024)

os.makedirs(OUTPUT_DIR, exist_ok=True)

sz = export_conll(train_set, os.path.join(OUTPUT_DIR, "train.conll"))
print(f"\n  ✓ train.conll ({sz:.2f} MB)")
sz = export_conll(dev_set, os.path.join(OUTPUT_DIR, "dev.conll"))
print(f"  ✓ dev.conll ({sz:.2f} MB)")
sz = export_conll(test_set, os.path.join(OUTPUT_DIR, "test.conll"))
print(f"  ✓ test.conll ({sz:.2f} MB)")
sz = export_conll(tier1, os.path.join(OUTPUT_DIR, "tier1_highconf.conll"))
print(f"  ✓ tier1_highconf.conll ({sz:.2f} MB)")
sz = export_conll(tier2, os.path.join(OUTPUT_DIR, "tier2_medconf.conll"))
print(f"  ✓ tier2_medconf.conll ({sz:.2f} MB)")
sz = export_conll(tier3, os.path.join(OUTPUT_DIR, "tier3_all.conll"))
print(f"  ✓ tier3_all.conll ({sz:.2f} MB)")

# Save final pkl
with open(projected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ {projected_pkl_path}")

# Save correction rules for reuse on large corpus
corrections_path = os.path.join(OUTPUT_DIR, "correction_rules.json")
serializable_corrections = {
    k: {"wrong_tags": list(v["wrong_tags"]), "correct_tag": v["correct_tag"]}
    for k, v in GLOBAL_CORRECTIONS.items()
}
with open(corrections_path, "w", encoding="utf-8") as f:
    json.dump(serializable_corrections, f, ensure_ascii=False, indent=2)
print(f"  ✓ {corrections_path}")

print("\n" + "=" * 60)
print("Cell 6g Complete.")
print("=" * 60)
print("\nPlease review the X-tagged tokens list above.")
print("Report any that should have a specific tag.")
print("\nAfter X-tag review, we proceed to:")
print("  Cell 7: Process large corpus (1.3M sentences)")
print("  Cell 8: Train BiLSTM-CRF PoS tagger")

Loaded 13,155 sentences

APPLYING ROUND 5 (FINAL) CORRECTIONS

  Total tokens: 157,146
  Total corrections: 1,811

  Top corrections:
    hman: NOUN→VERB                                  238
    hma: NOUN→ADJ                                    234
    lam: NOUN→ADJ                                    220
    zel: VERB→ADV                                    189
    nuam: VERB→ADJ                                   149
    lang: NOUN→VERB                                  144
    han: NOUN→ADV                                    109
    rei: NOUN→ADV                                    105
    lalpa: ADJ→NOUN                                  105
    thi: NOUN→VERB                                    70
    pek: NOUN→VERB                                    66
    pan: NOUN→VERB                                    61
    ngen: NOUN→VERB                                   41
    zai: NOUN→VERB                                    33
    takte: NOUN→ADV                                   24
    sawiho:

## Cell 6h: Apply X-Tag Corrections + Final Re-export

In [9]:
"""
Cell 6h: Apply X-Tag Corrections + Final Re-export
=====================================================
Applies user-provided tags for X-tagged tokens, then re-exports everything.
"""

import os
import pickle
import random
import json
import statistics
from collections import Counter

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")
small_mz_path = os.path.join(DATA_DIR, "small.mz")

# ============================================================
# Load data
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
small_mz = read_lines(small_mz_path)
print(f"Loaded {len(projected_corpus):,} sentences")

# ============================================================
# X-TAG CORRECTIONS from user review
# Format: clean_token -> tag
# For ambiguous words (bu, no, dàn, űr), we pick the more common usage
# ============================================================

X_TAG_FIXES = {
    "nu": "NOUN",
    "erawh": "CCONJ",
    "ṭhiante": "NOUN",
    "aia": "ADJ",
    "thinlung": "NOUN",
    "nân": "CCONJ",
    "anga": "ADJ",
    "nia": "PART",
    "zet": "ADV",
    "tal": "ADJ",
    "ruihhlo": "NOUN",
    "mei": "NOUN",
    "bo": "VERB",
    "amc": "NOUN",
    "bu": "NOUN",       # more common: nest
    "ni'": "PART",
    "ṭih": "VERB",
    "pawha": "CCONJ",
    "â": "ADJ",
    "ang'": "ADJ",
    "anih": "CCONJ",
    "chauha": "ADJ",
    "'ka": "PRON",
    "ina": "PART",
    "hemi": "ADJ",
    "isuan": "NOUN",
    "chanvo": "NOUN",
    "thlasik": "ADJ",
    "ati": "VERB",
    "awmze": "ADJ",
    "zeuh": "ADJ",
    "tâk": "ADV",
    "sorkar": "NOUN",
    "vêl": "ADJ",
    "hret": "ADJ",
    "hotute": "NOUN",
    "tik": "ADJ",
    "lohzia": "ADJ",
    "nên": "CCONJ",
    "samuela": "PROPN",
    "kalta": "ADV",
    "târ": "VERB",
    "berte": "ADJ",
    "vanram": "NOUN",
    "lo'": "ADJ",
    "mup": "NOUN",
    "no": "NOUN",       # more common: cup
    "lantir": "VERB",
    "hmuha": "ADV",
    "bera": "ADJ",
    "tom": "NOUN",       # Note: overrides the PROPN from R5 for X-tagged instances
    "bi": "ADJ",
    "thlap": "ADV",
    "keia": "PRON",
    "pelh": "ADV",
    "zaa": "ADJ",
    "mize": "ADJ",
    "deuha": "ADJ",
    "alo": "ADV",
    "vun": "NOUN",
    "saula": "PROPN",
    "dial": "ADJ",
    "tlema": "ADJ",
    "thumal": "NOUN",
    "kea": "ADJ",
    "ṭha'": "ADJ",
    "thute": "NOUN",
    "engemaw": "CCONJ",
    "tom-an": "NOUN",
    "bible": "NOUN",
    "ang,'": "ADJ",
    "hmuhsit": "VERB",
    "loh'": "ADJ",
    "a'": "PART",
    "chî": "NOUN",
    "thawka": "ADV",
    "lohva": "ADJ",
    "rawh'": "ADP",
    "kutke": "NOUN",
    "kuthnathawktute": "NOUN",
    "upate": "NOUN",
    "êm": "ADJ",
    "loa": "ADJ",
    "nun'": "NOUN",
    "u'": "PRON",
    "'kei": "PRON",
    "thianho": "NOUN",
    "dàn": "NOUN",       # more common: law
    "phd": "NOUN",
    "london": "PROPN",
    "'a": "PART",
    "dila": "ADJ",
    "minit": "ADJ",
    "emawa": "CCONJ",
    "tute": "PRON",
    "chhûngmu": "NOUN",
    "savate": "NOUN",
    "űr": "VERB",        # more common: boil/steam
    "ṭhutkhawm": "VERB",
    "intirh": "VERB",
    "nei'": "VERB",
    "lalpa'n": "NOUN",
    "mita": "ADJ",
    "hani": "PROPN",
    "ema": "ADJ",
    "ṭhensak": "VERB",
    "tansak": "VERB",
    "vânduai": "ADV",
    "âwl": "VERB",
    "chuta": "ADJ",
    "ropuia": "NOUN",
    "meizûk": "ADV",
    "ên": "VERB",
    "pulis": "NOUN",
    "siami": "PROPN",
    "sâp": "NOUN",
    "vin": "ADJ",
    "pêm": "VERB",
    "thawha": "ADV",
    "tehchiam": "ADJ",
    "tuifawnten": "NOUN",
    "york": "NOUN",
    "nulat": "ADV",
    "vui": "VERB",
    "ve'": "ADJ",
    "dauh": "ADJ",
    "kathy": "PROPN",
    "ṭhianho": "NOUN",
    "tama": "ADJ",
    "a'n": "PRON",
    "pe'": "VERB",
    "hlat": "VERB",
    "fc": "NOUN",
    "'thlarau": "NOUN",
    "meh": "VERB",
    "niha": "ADV",
    "ka'n": "PRON",
    "la'": "VERB",
    "senno": "ADJ",
    "chini": "NOUN",
    "danny": "PROPN",
    "hlap": "VERB",
    "abci": "NOUN",
    "ṭhîn'": "ADJ",
    "pianpun": "VERB",
    "tarmit": "NOUN",
    "elpui": "ADJ",
    "beha": "ADV",
    "boston": "NOUN",
    "linda": "PROPN",
    "thlitfim": "VERB",
    "hnu-a": "ADJ",
    "tv": "NOUN",
    "thlarau'": "NOUN",
    "pawnto": "VERB",
    "india": "NOUN",
    "samuelan": "PROPN",
    "tak'": "ADJ",
    "dik'": "ADJ",
    "hlân": "VERB",
    "nabin-a": "NOUN",
    "ṭuma": "ADV",
    "maia": "ADJ",
    "inte": "NOUN",
    "dante": "NOUN",
    "inlehtîr": "VERB",
    "tîr": "VERB",
    "french": "NOUN",
    "takin'": "ADV",
    "gps": "NOUN",
    "'isua": "NOUN",
    "ęm": "ADJ",
    "kauh": "ADV",
}

# ============================================================
# Apply X-tag fixes
# ============================================================
print("\n" + "=" * 60)
print("APPLYING X-TAG CORRECTIONS")
print("=" * 60)

fix_count = Counter()
total_tokens = 0
x_before = 0

for sent in projected_corpus:
    for tok in sent:
        total_tokens += 1
        if tok["final_tag"] == "X":
            x_before += 1
            # Try exact match first (including punctuation)
            token_lower = tok["token"].lower()
            clean = token_lower.rstrip(".,!?;:'\"()-")
            
            matched = False
            # Try exact token (for tokens with quotes/apostrophes like ni', a', etc.)
            if token_lower in X_TAG_FIXES:
                tok["final_tag"] = X_TAG_FIXES[token_lower]
                tok["confidence"] = 0.90
                tok["source"] = "human_x_fix"
                fix_count[f"{token_lower}→{tok['final_tag']}"] += 1
                matched = True
            # Try cleaned version
            elif clean in X_TAG_FIXES:
                tok["final_tag"] = X_TAG_FIXES[clean]
                tok["confidence"] = 0.90
                tok["source"] = "human_x_fix"
                fix_count[f"{clean}→{tok['final_tag']}"] += 1
                matched = True
            # Try with leading quote stripped
            elif token_lower.lstrip("'\"") in X_TAG_FIXES:
                stripped = token_lower.lstrip("'\"")
                tok["final_tag"] = X_TAG_FIXES[stripped]
                tok["confidence"] = 0.90
                tok["source"] = "human_x_fix"
                fix_count[f"{stripped}→{tok['final_tag']}"] += 1
                matched = True

# Also apply the previous global corrections one more time to catch stragglers
# Load correction rules
corrections_path = os.path.join(OUTPUT_DIR, "correction_rules.json")
if os.path.exists(corrections_path):
    with open(corrections_path, "r") as f:
        saved_rules = json.load(f)
    
    for sent in projected_corpus:
        for tok in sent:
            clean = tok["token"].lower().rstrip(".,!?;:'\"()-")
            if clean in saved_rules:
                rule = saved_rules[clean]
                if tok["final_tag"] in rule["wrong_tags"]:
                    tok["final_tag"] = rule["correct_tag"]
                    tok["confidence"] = 0.90
                    tok["source"] = "human_correction_final"
                    fix_count[f"global:{clean}→{rule['correct_tag']}"] += 1

x_after = sum(1 for sent in projected_corpus for tok in sent if tok["final_tag"] == "X")

print(f"\n  X-tagged before: {x_before:,}")
print(f"  X-tagged after:  {x_after:,}")
print(f"  X tags resolved: {x_before - x_after:,}")
print(f"  Total fixes applied: {sum(fix_count.values()):,}")

print(f"\n  Top X-tag fixes:")
for fix, count in fix_count.most_common(30):
    print(f"    {fix:<40} {count:>5}")

# ============================================================
# FINAL STATISTICS
# ============================================================
print("\n" + "=" * 60)
print("FINAL CORPUS STATISTICS (after all corrections)")
print("=" * 60)

final_dist = Counter()
source_dist = Counter()
conf_list = []

for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1
        source_dist[tok["source"]] += 1
        conf_list.append(tok["confidence"])

print(f"\n  Tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

print(f"\n  Confidence: mean={statistics.mean(conf_list):.3f}, "
      f"median={statistics.median(conf_list):.3f}")
print(f"  >=0.90: {sum(1 for c in conf_list if c >= 0.90):>8,} ({100*sum(1 for c in conf_list if c >= 0.90)/total_tokens:.1f}%)")
print(f"  >=0.70: {sum(1 for c in conf_list if c >= 0.70):>8,} ({100*sum(1 for c in conf_list if c >= 0.70)/total_tokens:.1f}%)")

# Remaining X tokens
if x_after > 0:
    remaining_x = Counter()
    for sent in projected_corpus:
        for tok in sent:
            if tok["final_tag"] == "X":
                remaining_x[tok["token"].lower().rstrip(".,!?;:'\"()-")] += 1
    print(f"\n  Remaining X tokens ({x_after} total, {len(remaining_x)} unique):")
    for t, c in remaining_x.most_common(20):
        print(f"    {t:<30} {c:>3}")

# ============================================================
# RE-EXPORT EVERYTHING
# ============================================================
print("\n" + "=" * 60)
print("RE-EXPORTING FINAL TRAINING DATA")
print("=" * 60)

# Rebuild tiers
tier1, tier2, tier3 = [], [], []

for sent_idx, sent in enumerate(projected_corpus):
    token_tags = [(tok["token"], tok["final_tag"]) for tok in sent]
    min_conf = min(tok["confidence"] for tok in sent)
    mean_conf = statistics.mean(tok["confidence"] for tok in sent)
    
    entry = {
        "idx": sent_idx,
        "tokens": token_tags,
        "min_conf": min_conf,
        "mean_conf": mean_conf,
        "en_text": small_en[sent_idx] if sent_idx < len(small_en) else "",
    }
    
    tier3.append(entry)
    if min_conf >= 0.50:
        tier2.append(entry)
    if min_conf >= 0.70:
        tier1.append(entry)

print(f"  Tier 1 (high, min_conf>=0.70): {len(tier1):>8,}")
print(f"  Tier 2 (med,  min_conf>=0.50): {len(tier2):>8,}")
print(f"  Tier 3 (all):                  {len(tier3):>8,}")

# Train/Dev/Test
random.seed(42)
tier1_shuffled = tier1.copy()
random.shuffle(tier1_shuffled)

n_dev = max(500, len(tier1_shuffled) // 10)
n_test = max(500, len(tier1_shuffled) // 10)
if n_dev + n_test > len(tier1_shuffled):
    n_dev = len(tier1_shuffled) // 3
    n_test = len(tier1_shuffled) // 3

test_set = tier1_shuffled[:n_test]
dev_set = tier1_shuffled[n_test:n_test + n_dev]
dev_test_indices = set(s["idx"] for s in dev_set + test_set)
train_set = [s for s in tier2 if s["idx"] not in dev_test_indices]

train_tokens = sum(len(s["tokens"]) for s in train_set)
dev_tokens = sum(len(s["tokens"]) for s in dev_set)
test_tokens = sum(len(s["tokens"]) for s in test_set)

print(f"\n  Train: {len(train_set):>8,} sentences ({train_tokens:,} tokens)")
print(f"  Dev:   {len(dev_set):>8,} sentences ({dev_tokens:,} tokens)")
print(f"  Test:  {len(test_set):>8,} sentences ({test_tokens:,} tokens)")

# Tag distribution check
train_tags = Counter(tag for s in train_set for _, tag in s["tokens"])
dev_tags = Counter(tag for s in dev_set for _, tag in s["tokens"])
test_tags = Counter(tag for s in test_set for _, tag in s["tokens"])

all_tags = sorted(set(train_tags) | set(dev_tags) | set(test_tags))
print(f"\n  {'Tag':<10} {'Train%':>8} {'Dev%':>8} {'Test%':>8}")
for tag in all_tags:
    t = 100 * train_tags[tag] / train_tokens if train_tokens else 0
    d = 100 * dev_tags[tag] / dev_tokens if dev_tokens else 0
    te = 100 * test_tags[tag] / test_tokens if test_tokens else 0
    print(f"  {tag:<10} {t:>7.1f}% {d:>7.1f}% {te:>7.1f}%")

# Export
def export_conll(sentences, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for sent in sentences:
            for token, tag in sent["tokens"]:
                f.write(f"{token}\t{tag}\n")
            f.write("\n")
    return os.path.getsize(filepath) / (1024 * 1024)

os.makedirs(OUTPUT_DIR, exist_ok=True)

files_exported = []
for name, data in [("train.conll", train_set), ("dev.conll", dev_set), 
                    ("test.conll", test_set), ("tier1_highconf.conll", tier1),
                    ("tier2_medconf.conll", tier2), ("tier3_all.conll", tier3)]:
    path = os.path.join(OUTPUT_DIR, name)
    sz = export_conll(data, path)
    print(f"  ✓ {name} ({sz:.2f} MB)")
    files_exported.append(path)

# Save final pkl
with open(projected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ projected_corpus_final.pkl")

# Update correction rules with X-tag fixes included
all_corrections = {}
if os.path.exists(corrections_path):
    with open(corrections_path, "r") as f:
        all_corrections = json.load(f)

# Add X-tag fixes as corrections (for reuse on large corpus)
for token, tag in X_TAG_FIXES.items():
    clean = token.lower().rstrip(".,!?;:'\"()-")
    if clean not in all_corrections:
        all_corrections[clean] = {"wrong_tags": ["X"], "correct_tag": tag}
    else:
        if "X" not in all_corrections[clean]["wrong_tags"]:
            all_corrections[clean]["wrong_tags"].append("X")

with open(corrections_path, "w", encoding="utf-8") as f:
    json.dump(all_corrections, f, ensure_ascii=False, indent=2)
print(f"  ✓ correction_rules.json (updated with {len(all_corrections)} rules)")

# Save dataset stats
stats = {
    "total_sentences": len(projected_corpus),
    "total_tokens": total_tokens,
    "x_remaining": x_after,
    "tier1": len(tier1),
    "tier2": len(tier2), 
    "tier3": len(tier3),
    "train_sentences": len(train_set),
    "dev_sentences": len(dev_set),
    "test_sentences": len(test_set),
    "train_tokens": train_tokens,
    "dev_tokens": dev_tokens,
    "test_tokens": test_tokens,
    "tag_distribution": dict(final_dist.most_common()),
    "correction_rounds": 5,
    "total_correction_rules": len(all_corrections),
}
with open(os.path.join(OUTPUT_DIR, "dataset_stats_final.json"), "w") as f:
    json.dump(stats, f, indent=2)
print(f"  ✓ dataset_stats_final.json")

print("\n" + "=" * 60)
print("Cell 6h Complete — ALL CORRECTIONS APPLIED")
print("=" * 60)
print(f"\nSummary:")
print(f"  5 rounds of human review completed")
print(f"  {len(all_corrections)} correction rules accumulated")
print(f"  X tags reduced from {x_before} to {x_after}")
print(f"  Training data ready for model training")
print(f"\nNext steps:")
print(f"  Cell 7: Process large corpus (1.3M sentences) for data augmentation")
print(f"  Cell 8: Train BiLSTM-CRF PoS tagger")

Loaded 13,155 sentences

APPLYING X-TAG CORRECTIONS

  X-tagged before: 1,884
  X-tagged after:  772
  X tags resolved: 1,112
  Total fixes applied: 1,112

  Top X-tag fixes:
    nu→NOUN                                    111
    erawh→CCONJ                                108
    ṭhiante→NOUN                                71
    aia→ADJ                                     48
    thinlung→NOUN                               48
    nân→CCONJ                                   25
    anga→ADJ                                    25
    nia→PART                                    24
    zet→ADV                                     23
    tal→ADJ                                     23
    ruihhlo→NOUN                                23
    mei→NOUN                                    21
    bo→VERB                                     20
    amc→NOUN                                    18
    bu→NOUN                                     16
    ṭih→VERB                                    14
    pawha

## Cell 6i: Context-Based X-Tag Resolution

In [11]:
"""
Cell 6i: Quick Automatic X-Tag Fix + Move On
===============================================
Fix what we can automatically, leave the rest as X.
772 X tokens (0.5%) won't hurt training significantly.
"""

import os
import pickle
import json
import statistics
import random
from collections import Counter

# ============================================================
# PATHS
# ============================================================
OUTPUT_DIR = "data2"
DATA_DIR = "data"

projected_pkl_path = os.path.join(OUTPUT_DIR, "projected_corpus_final.pkl")
small_en_path = os.path.join(DATA_DIR, "small.en")
small_mz_path = os.path.join(DATA_DIR, "small.mz")

# ============================================================
# Load
# ============================================================
with open(projected_pkl_path, "rb") as f:
    projected_corpus = pickle.load(f)

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

small_en = read_lines(small_en_path)
small_mz = read_lines(small_mz_path)
print(f"Loaded {len(projected_corpus):,} sentences")

# ============================================================
# Previous X-tag fixes (reload for apostrophe normalization)
# ============================================================
X_TAG_FIXES = {
    "thla": "NOUN",       # most common: month/feather/wing
    "ni'": "PART",
    "ang'": "ADJ",
    "'ka": "PRON",
    "lo'": "ADJ",
    "ṭha'": "ADJ",
    "ang,'": "ADJ",
    "loh'": "ADJ",
    "a'": "PART",
    "rawh'": "ADP",
    "nun'": "NOUN",
    "u'": "PRON",
    "'kei": "PRON",
    "'a": "PART",
    "nei'": "VERB",
    "ṭhîn'": "ADJ",
    "tak'": "ADJ",
    "dik'": "ADJ",
    "ve'": "ADJ",
    "la'": "VERB",
    "pe'": "VERB",
    "takin'": "ADV",
    "'isua": "NOUN",
    "'thlarau": "NOUN",
    "thlarau'": "NOUN",
    "ka'n": "PRON",
    "a'n": "PRON",
}

# ============================================================
# Apply fixes with Unicode normalization
# ============================================================
print("\n" + "=" * 60)
print("AUTOMATIC X-TAG FIXES")
print("=" * 60)

fix_count = Counter()
x_before = sum(1 for s in projected_corpus for t in s if t["final_tag"] == "X")

def normalize_quotes(text):
    """Normalize various quote characters to ASCII."""
    replacements = {
        '\u2018': "'",  # left single quote
        '\u2019': "'",  # right single quote
        '\u201C': '"',  # left double quote
        '\u201D': '"',  # right double quote
        '\u0060': "'",  # backtick
        '\u00B4': "'",  # acute accent
        '\u2032': "'",  # prime
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

for sent in projected_corpus:
    for tok in sent:
        if tok["final_tag"] == "X":
            raw = tok["token"]
            
            # Normalize quotes in the token for matching
            normalized = normalize_quotes(raw.lower())
            clean = normalized.rstrip(".,!?;:'\"()-")
            clean_lstrip = normalized.lstrip("'\"")
            clean_both = normalized.lstrip("'\"").rstrip(".,!?;:'\"()-")
            
            # Try multiple matching strategies
            matched = False
            for candidate in [normalized, clean, clean_lstrip, clean_both, raw.lower()]:
                if candidate in X_TAG_FIXES:
                    tok["final_tag"] = X_TAG_FIXES[candidate]
                    tok["confidence"] = 0.85
                    tok["source"] = "auto_x_fix"
                    fix_count[f"{candidate}→{tok['final_tag']}"] += 1
                    matched = True
                    break
            
            # If still X and it's a single character, try common defaults
            if not matched and len(clean_both) <= 1:
                # Single letters/characters that are X are likely noise
                # Leave as X
                pass
            
            # If still X and looks like a word (not just punctuation/numbers)
            # Apply most-likely-tag heuristic based on common patterns
            if not matched and tok["final_tag"] == "X":
                # Words ending in common Mizo suffixes
                if clean_both.endswith("na") and len(clean_both) > 3:
                    tok["final_tag"] = "NOUN"
                    tok["confidence"] = 0.70
                    tok["source"] = "auto_suffix_na"
                    fix_count["suffix_na→NOUN"] += 1
                elif clean_both.endswith("tu") and len(clean_both) > 3:
                    tok["final_tag"] = "NOUN"
                    tok["confidence"] = 0.70
                    tok["source"] = "auto_suffix_tu"
                    fix_count["suffix_tu→NOUN"] += 1
                elif clean_both.endswith("in") and len(clean_both) > 3:
                    tok["final_tag"] = "ADV"
                    tok["confidence"] = 0.65
                    tok["source"] = "auto_suffix_in"
                    fix_count["suffix_in→ADV"] += 1
                elif clean_both.endswith("ah") and len(clean_both) > 3:
                    tok["final_tag"] = "ADV"
                    tok["confidence"] = 0.65
                    tok["source"] = "auto_suffix_ah"
                    fix_count["suffix_ah→ADV"] += 1
                elif clean_both.endswith("te") and len(clean_both) > 4:
                    tok["final_tag"] = "NOUN"
                    tok["confidence"] = 0.70
                    tok["source"] = "auto_suffix_te"
                    fix_count["suffix_te→NOUN"] += 1

x_after = sum(1 for s in projected_corpus for t in s if t["final_tag"] == "X")

print(f"\n  X before: {x_before}")
print(f"  X after:  {x_after}")
print(f"  Fixed:    {x_before - x_after}")
print(f"\n  Fix breakdown:")
for fix, count in fix_count.most_common():
    print(f"    {fix:<40} {count:>5}")

# ============================================================
# Final stats
# ============================================================
print("\n" + "=" * 60)
print("FINAL CORPUS STATISTICS")
print("=" * 60)

total_tokens = sum(len(s) for s in projected_corpus)
final_dist = Counter()
for sent in projected_corpus:
    for tok in sent:
        final_dist[tok["final_tag"]] += 1

print(f"\n  Total tokens: {total_tokens:,}")
print(f"  Tag distribution:")
for tag, count in final_dist.most_common():
    print(f"    {tag:<10} {count:>8,} ({100*count/total_tokens:>5.1f}%)")

print(f"\n  X remaining: {x_after} ({100*x_after/total_tokens:.2f}%)")

# ============================================================
# Re-export final data
# ============================================================
print("\n" + "=" * 60)
print("RE-EXPORTING FINAL DATA")
print("=" * 60)

# Save pkl
with open(projected_pkl_path, "wb") as f:
    pickle.dump(projected_corpus, f)
print(f"  ✓ projected_corpus_final.pkl")

# Rebuild tiers and splits
tier1, tier2, tier3 = [], [], []
for sent_idx, sent in enumerate(projected_corpus):
    token_tags = [(tok["token"], tok["final_tag"]) for tok in sent]
    min_conf = min(tok["confidence"] for tok in sent)
    mean_conf = statistics.mean(tok["confidence"] for tok in sent)
    entry = {
        "idx": sent_idx,
        "tokens": token_tags,
        "min_conf": min_conf,
        "mean_conf": mean_conf,
    }
    tier3.append(entry)
    if min_conf >= 0.50:
        tier2.append(entry)
    if min_conf >= 0.70:
        tier1.append(entry)

random.seed(42)
tier1_shuffled = tier1.copy()
random.shuffle(tier1_shuffled)

n_dev = max(500, len(tier1_shuffled) // 10)
n_test = max(500, len(tier1_shuffled) // 10)
if n_dev + n_test > len(tier1_shuffled):
    n_dev = len(tier1_shuffled) // 3
    n_test = len(tier1_shuffled) // 3

test_set = tier1_shuffled[:n_test]
dev_set = tier1_shuffled[n_test:n_test + n_dev]
dev_test_indices = set(s["idx"] for s in dev_set + test_set)
train_set = [s for s in tier2 if s["idx"] not in dev_test_indices]

def export_conll(sentences, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for sent in sentences:
            for token, tag in sent["tokens"]:
                f.write(f"{token}\t{tag}\n")
            f.write("\n")
    return os.path.getsize(filepath) / (1024 * 1024)

train_tokens = sum(len(s["tokens"]) for s in train_set)
dev_tokens = sum(len(s["tokens"]) for s in dev_set)
test_tokens = sum(len(s["tokens"]) for s in test_set)

print(f"\n  Tier 1: {len(tier1):,} | Tier 2: {len(tier2):,} | Tier 3: {len(tier3):,}")
print(f"  Train: {len(train_set):,} sent ({train_tokens:,} tok)")
print(f"  Dev:   {len(dev_set):,} sent ({dev_tokens:,} tok)")
print(f"  Test:  {len(test_set):,} sent ({test_tokens:,} tok)")

for name, data in [("train.conll", train_set), ("dev.conll", dev_set),
                    ("test.conll", test_set), ("tier1_highconf.conll", tier1),
                    ("tier2_medconf.conll", tier2), ("tier3_all.conll", tier3)]:
    sz = export_conll(data, os.path.join(OUTPUT_DIR, name))
    print(f"  ✓ {name} ({sz:.2f} MB)")

print("\n" + "=" * 60)
print("Cell 6i Complete — Ready for Cell 7 (Large Corpus)")
print("=" * 60)

Loaded 13,155 sentences

AUTOMATIC X-TAG FIXES

  X before: 772
  X after:  527
  Fixed:    245

  Fix breakdown:
    thla→NOUN                                   95
    suffix_te→NOUN                              37
    ni'→PART                                    15
    ang'→ADJ                                    11
    suffix_na→NOUN                              10
    'ka→PRON                                     9
    suffix_in→ADV                                6
    lo'→ADJ                                      6
    'a→PART                                      4
    ṭha'→ADJ                                     4
    ang,'→ADJ                                    3
    loh'→ADJ                                     3
    a'→PART                                      3
    rawh'→ADP                                    3
    nun'→NOUN                                    3
    u'→PRON                                      3
    'kei→PRON                                    3
    nei'→VERB      

## Cell 7a: Filter Large Corpus for High-Quality Alignment

In [12]:
"""
Cell 7a: Filter Large Corpus for High-Quality Alignment
=========================================================
Filter criteria for good PoS projection via alignment:
1. Sentence length: not too short, not too long (5-20 words)
2. Length ratio: MZ/EN should be close to 1.0 (0.5 - 2.0)
3. Low punctuation/special character ratio
4. No excessive apostrophes or quotes
5. No empty or near-empty sentences
6. Mostly alphabetic content (not numbers/codes)
7. Both MZ and EN should be actual sentences (not fragments)
"""

import os
import re
import time
from collections import Counter

# ============================================================
# PATHS
# ============================================================
DATA_DIR = "data"
OUTPUT_DIR = "data2"

large_mz_path = os.path.join(DATA_DIR, "large.mz")
large_en_path = os.path.join(DATA_DIR, "large.en")

# ============================================================
# Load large corpus
# ============================================================
print("Loading large corpus...")

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

large_mz = read_lines(large_mz_path)
large_en = read_lines(large_en_path)

print(f"  Loaded {len(large_mz):,} Mizo sentences")
print(f"  Loaded {len(large_en):,} English sentences")

# ============================================================
# Define filtering criteria
# ============================================================

def count_punct(text):
    """Count punctuation characters in text."""
    return sum(1 for c in text if c in '.,!?;:\'"()-–—…/\\&*#@%+=<>~^|`[]{}')

def count_apostrophes(text):
    """Count apostrophe/quote characters."""
    return sum(1 for c in text if c in "'\u2018\u2019\u201C\u201D\"`")

def count_alpha_words(tokens):
    """Count tokens that are mostly alphabetic."""
    return sum(1 for t in tokens if re.match(r'^[a-zA-ZṬṭÂâÊêÎîÔôÛûĂăĄąĈĉḌḍŊŋ]+[.,!?;:]*$', t))

def has_repeated_tokens(tokens, threshold=3):
    """Check if any token repeats more than threshold times."""
    counts = Counter(t.lower() for t in tokens)
    return any(c > threshold for c in counts.values())

def analyze_pair(mz_line, en_line):
    """Analyze a sentence pair and return quality metrics."""
    mz_tokens = mz_line.split()
    en_tokens = en_line.split()
    
    mz_len = len(mz_tokens)
    en_len = len(en_tokens)
    
    # Length ratio
    ratio = mz_len / en_len if en_len > 0 else 999
    
    # Punctuation ratio
    mz_punct_ratio = count_punct(mz_line) / len(mz_line) if mz_line else 1
    en_punct_ratio = count_punct(en_line) / len(en_line) if en_line else 1
    
    # Apostrophe count
    mz_apost = count_apostrophes(mz_line)
    en_apost = count_apostrophes(en_line)
    
    # Alpha word ratio
    mz_alpha = count_alpha_words(mz_tokens) / mz_len if mz_len > 0 else 0
    en_alpha = count_alpha_words(en_tokens) / en_len if en_len > 0 else 0
    
    return {
        "mz_len": mz_len,
        "en_len": en_len,
        "ratio": ratio,
        "mz_punct_ratio": mz_punct_ratio,
        "en_punct_ratio": en_punct_ratio,
        "mz_apost": mz_apost,
        "en_apost": en_apost,
        "mz_alpha": mz_alpha,
        "en_alpha": en_alpha,
    }

# ============================================================
# Analyze ALL pairs (quick pass to determine thresholds)
# ============================================================
print("\n" + "=" * 60)
print("ANALYZING LARGE CORPUS QUALITY")
print("=" * 60)

start_time = time.time()

# Quick stats
mz_lengths = []
en_lengths = []
ratios = []
mz_punct_ratios = []
mz_apost_counts = []
mz_alpha_ratios = []

for i in range(len(large_mz)):
    mz_tokens = large_mz[i].split()
    en_tokens = large_en[i].split()
    
    mz_len = len(mz_tokens)
    en_len = len(en_tokens)
    
    mz_lengths.append(mz_len)
    en_lengths.append(en_len)
    
    if en_len > 0:
        ratios.append(mz_len / en_len)
    
    if large_mz[i]:
        mz_punct_ratios.append(count_punct(large_mz[i]) / len(large_mz[i]))
        mz_apost_counts.append(count_apostrophes(large_mz[i]))
        mz_alpha_ratios.append(
            count_alpha_words(mz_tokens) / mz_len if mz_len > 0 else 0
        )

elapsed = time.time() - start_time
print(f"  Analysis complete in {elapsed:.1f}s")

import statistics

print(f"\n  MZ sentence length: mean={statistics.mean(mz_lengths):.1f}, "
      f"median={statistics.median(mz_lengths):.0f}, "
      f"min={min(mz_lengths)}, max={max(mz_lengths)}")
print(f"  EN sentence length: mean={statistics.mean(en_lengths):.1f}, "
      f"median={statistics.median(en_lengths):.0f}, "
      f"min={min(en_lengths)}, max={max(en_lengths)}")
print(f"  MZ/EN ratio: mean={statistics.mean(ratios):.2f}, "
      f"median={statistics.median(ratios):.2f}")

# ============================================================
# Show what different filter thresholds yield
# ============================================================
print("\n" + "=" * 60)
print("FILTER THRESHOLD ANALYSIS")
print("=" * 60)

# Define filter sets to test
filter_configs = [
    {
        "name": "Loose (5-25 words, ratio 0.4-2.5)",
        "mz_min": 5, "mz_max": 25,
        "en_min": 5, "en_max": 25,
        "ratio_min": 0.4, "ratio_max": 2.5,
        "max_punct_ratio": 0.15,
        "max_apost": 3,
        "min_alpha_ratio": 0.70,
    },
    {
        "name": "Moderate (5-20 words, ratio 0.5-2.0)",
        "mz_min": 5, "mz_max": 20,
        "en_min": 5, "en_max": 20,
        "ratio_min": 0.5, "ratio_max": 2.0,
        "max_punct_ratio": 0.10,
        "max_apost": 2,
        "min_alpha_ratio": 0.80,
    },
    {
        "name": "Strict (6-15 words, ratio 0.6-1.7)",
        "mz_min": 6, "mz_max": 15,
        "en_min": 6, "en_max": 15,
        "ratio_min": 0.6, "ratio_max": 1.7,
        "max_punct_ratio": 0.08,
        "max_apost": 1,
        "min_alpha_ratio": 0.85,
    },
]

for config in filter_configs:
    passed = 0
    for i in range(len(large_mz)):
        mz_tokens = large_mz[i].split()
        en_tokens = large_en[i].split()
        mz_len = len(mz_tokens)
        en_len = len(en_tokens)
        
        if mz_len < config["mz_min"] or mz_len > config["mz_max"]:
            continue
        if en_len < config["en_min"] or en_len > config["en_max"]:
            continue
        
        ratio = mz_len / en_len if en_len > 0 else 999
        if ratio < config["ratio_min"] or ratio > config["ratio_max"]:
            continue
        
        mz_punct = count_punct(large_mz[i]) / len(large_mz[i]) if large_mz[i] else 1
        if mz_punct > config["max_punct_ratio"]:
            continue
        
        mz_apost = count_apostrophes(large_mz[i])
        if mz_apost > config["max_apost"]:
            continue
        
        mz_alpha = count_alpha_words(mz_tokens) / mz_len if mz_len > 0 else 0
        if mz_alpha < config["min_alpha_ratio"]:
            continue
        
        passed += 1
    
    print(f"\n  {config['name']}:")
    print(f"    Passed: {passed:>10,} / {len(large_mz):,} ({100*passed/len(large_mz):.1f}%)")
    
    # Estimate processing time (based on small corpus: ~10.6 pairs/sec for SimAlign)
    est_align_hours = passed / (10.6 * 3600)
    print(f"    Est. alignment time: ~{est_align_hours:.1f} hours")

# ============================================================
# Apply MODERATE filter and export
# ============================================================
print("\n" + "=" * 60)
print("APPLYING MODERATE FILTER")
print("=" * 60)

config = filter_configs[1]  # Moderate
filtered_indices = []
rejection_reasons = Counter()

for i in range(len(large_mz)):
    mz_tokens = large_mz[i].split()
    en_tokens = large_en[i].split()
    mz_len = len(mz_tokens)
    en_len = len(en_tokens)
    
    # Check each criterion
    if mz_len < config["mz_min"] or mz_len > config["mz_max"]:
        rejection_reasons["mz_length"] += 1
        continue
    if en_len < config["en_min"] or en_len > config["en_max"]:
        rejection_reasons["en_length"] += 1
        continue
    
    ratio = mz_len / en_len if en_len > 0 else 999
    if ratio < config["ratio_min"] or ratio > config["ratio_max"]:
        rejection_reasons["length_ratio"] += 1
        continue
    
    mz_punct = count_punct(large_mz[i]) / len(large_mz[i]) if large_mz[i] else 1
    if mz_punct > config["max_punct_ratio"]:
        rejection_reasons["punctuation"] += 1
        continue
    
    mz_apost = count_apostrophes(large_mz[i])
    if mz_apost > config["max_apost"]:
        rejection_reasons["apostrophes"] += 1
        continue
    
    mz_alpha = count_alpha_words(mz_tokens) / mz_len if mz_len > 0 else 0
    if mz_alpha < config["min_alpha_ratio"]:
        rejection_reasons["non_alpha"] += 1
        continue
    
    filtered_indices.append(i)

print(f"\n  Passed: {len(filtered_indices):,} / {len(large_mz):,} ({100*len(filtered_indices)/len(large_mz):.1f}%)")
print(f"\n  Rejection reasons:")
for reason, count in rejection_reasons.most_common():
    print(f"    {reason:<20} {count:>10,} ({100*count/len(large_mz):.1f}%)")

# Export filtered parallel files
filtered_mz_path = os.path.join(OUTPUT_DIR, "large_filtered.mz")
filtered_en_path = os.path.join(OUTPUT_DIR, "large_filtered.en")
filtered_idx_path = os.path.join(OUTPUT_DIR, "large_filtered_indices.txt")

with open(filtered_mz_path, "w", encoding="utf-8") as f:
    for idx in filtered_indices:
        f.write(large_mz[idx] + "\n")

with open(filtered_en_path, "w", encoding="utf-8") as f:
    for idx in filtered_indices:
        f.write(large_en[idx] + "\n")

with open(filtered_idx_path, "w", encoding="utf-8") as f:
    for idx in filtered_indices:
        f.write(str(idx) + "\n")

mz_size = os.path.getsize(filtered_mz_path) / (1024 * 1024)
en_size = os.path.getsize(filtered_en_path) / (1024 * 1024)

print(f"\n  ✓ {filtered_mz_path} ({mz_size:.2f} MB)")
print(f"  ✓ {filtered_en_path} ({en_size:.2f} MB)")
print(f"  ✓ {filtered_idx_path}")

# Show samples
print(f"\n  --- Filtered Samples (first 5) ---")
for i in range(min(5, len(filtered_indices))):
    idx = filtered_indices[i]
    print(f"    [{idx}] MZ: {large_mz[idx][:80]}")
    print(f"          EN: {large_en[idx][:80]}")
    print()

# Estimate times
est_hours = len(filtered_indices) / (10.6 * 3600)
print(f"\n  Estimated SimAlign time: ~{est_hours:.1f} hours")
print(f"  Estimated spaCy tagging: ~{len(filtered_indices)/392/60:.0f} minutes")

print("\n" + "=" * 60)
print("Cell 7a Complete.")
print("=" * 60)
print(f"\nFiltered {len(filtered_indices):,} high-quality sentence pairs.")
print("Next: Cell 7b will tag English + align + project (like small corpus).")
print("\nIf the estimated time is too long, we can:")
print("  Option A: Use 'Strict' filter for fewer sentences")
print("  Option B: Process in batches (e.g., 100K at a time)")
print("  Option C: Sample a random subset (e.g., 200K)")

Loading large corpus...
  Loaded 1,357,838 Mizo sentences
  Loaded 1,357,838 English sentences

ANALYZING LARGE CORPUS QUALITY
  Analysis complete in 20.4s

  MZ sentence length: mean=10.8, median=10, min=3, max=33
  EN sentence length: mean=9.8, median=9, min=4, max=48
  MZ/EN ratio: mean=1.16, median=1.12

FILTER THRESHOLD ANALYSIS

  Loose (5-25 words, ratio 0.4-2.5):
    Passed:  1,246,825 / 1,357,838 (91.8%)
    Est. alignment time: ~32.7 hours

  Moderate (5-20 words, ratio 0.5-2.0):
    Passed:  1,179,901 / 1,357,838 (86.9%)
    Est. alignment time: ~30.9 hours

  Strict (6-15 words, ratio 0.6-1.7):
    Passed:    837,860 / 1,357,838 (61.7%)
    Est. alignment time: ~22.0 hours

APPLYING MODERATE FILTER

  Passed: 1,179,901 / 1,357,838 (86.9%)

  Rejection reasons:
    en_length                83,153 (6.1%)
    mz_length                75,425 (5.6%)
    non_alpha                12,465 (0.9%)
    length_ratio              3,935 (0.3%)
    punctuation               2,397 (0.2%)
  

## Cell 7b: Process 150K Large Corpus Subset (Full Pipeline)

In [14]:
"""
Cell 7b: Process 100K Large Corpus Subset (Full Pipeline)
===========================================================
1. Random sample 100K from filtered large corpus
2. Tag English with spaCy
3. Align with SimAlign
4. Project tags onto Mizo
5. Apply lexicon + correction rules
6. Export augmented training data

Estimated time: ~30 min spaCy + ~2.5 hours SimAlign = ~3 hours total
"""

import os
import json
import pickle
import random
import time
import re
from collections import Counter, defaultdict

# ============================================================
# PATHS
# ============================================================
DATA_DIR = "data"
OUTPUT_DIR = "data2"

filtered_mz_path = os.path.join(OUTPUT_DIR, "large_filtered.mz")
filtered_en_path = os.path.join(OUTPUT_DIR, "large_filtered.en")
words_clean_path = os.path.join(OUTPUT_DIR, "mizo_words_clean.tsv")
phrases_clean_path = os.path.join(OUTPUT_DIR, "mizo_phrases_clean.tsv")
corrections_path = os.path.join(OUTPUT_DIR, "correction_rules.json")

# ============================================================
# Load resources
# ============================================================
print("Loading resources...")

def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

filtered_mz = read_lines(filtered_mz_path)
filtered_en = read_lines(filtered_en_path)
print(f"  Filtered corpus: {len(filtered_mz):,} sentence pairs")

# Load lexicon
word_lexicon = defaultdict(Counter)
with open(words_clean_path, "r", encoding="utf-8") as f:
    next(f)
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            word_lexicon[parts[0].lower()][parts[1]] += 1

phrase_lexicon = defaultdict(Counter)
with open(phrases_clean_path, "r", encoding="utf-8") as f:
    next(f)
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            phrase_lexicon[parts[0].lower()][parts[1]] += 1

print(f"  Word lexicon: {len(word_lexicon):,}")
print(f"  Phrase lexicon: {len(phrase_lexicon):,}")

# Load correction rules
with open(corrections_path, "r", encoding="utf-8") as f:
    correction_rules = json.load(f)
print(f"  Correction rules: {len(correction_rules)}")

# ============================================================
# Sample 100K random sentences
# ============================================================
SAMPLE_SIZE = 200_000

random.seed(42)
all_indices = list(range(len(filtered_mz)))
random.shuffle(all_indices)
sample_indices = sorted(all_indices[:SAMPLE_SIZE])

sample_mz = [filtered_mz[i] for i in sample_indices]
sample_en = [filtered_en[i] for i in sample_indices]

print(f"\n  Sampled {len(sample_mz):,} sentence pairs for full pipeline")

# ============================================================
# STEP 1: Tag English with spaCy
# ============================================================
print("\n" + "=" * 60)
print("STEP 1: ENGLISH PoS TAGGING (spaCy)")
print("=" * 60)

import spacy
nlp = spacy.load("en_core_web_sm")
print(f"  spaCy loaded: en_core_web_sm")

print(f"  Tagging {len(sample_en):,} English sentences...")
start_time = time.time()

tagged_english = []
batch_size = 1000

for doc in nlp.pipe(sample_en, batch_size=batch_size, n_process=1):
    sent_tokens = []
    for token in doc:
        sent_tokens.append({
            "token": token.text,
            "pos": token.pos_,
        })
    tagged_english.append(sent_tokens)
    
    if len(tagged_english) % 20000 == 0:
        elapsed = time.time() - start_time
        rate = len(tagged_english) / elapsed
        remaining = (len(sample_en) - len(tagged_english)) / rate
        print(f"    Tagged {len(tagged_english):>8,}/{len(sample_en):,} "
              f"({100*len(tagged_english)/len(sample_en):.1f}%) "
              f"[{elapsed:.0f}s, ~{remaining:.0f}s remaining]")

elapsed = time.time() - start_time
print(f"\n  ✓ English tagging complete in {elapsed:.0f}s ({len(sample_en)/elapsed:.0f} sent/sec)")

# Save English tags (in case we need to restart from alignment step)
en_tagged_path = os.path.join(OUTPUT_DIR, "large_sample_en_tagged.pkl")
with open(en_tagged_path, "wb") as f:
    pickle.dump(tagged_english, f)
print(f"  ✓ Saved: {en_tagged_path}")

# Also save sample sentences for potential restart
sample_path = os.path.join(OUTPUT_DIR, "large_sample_sentences.pkl")
with open(sample_path, "wb") as f:
    pickle.dump({"mz": sample_mz, "en": sample_en, "indices": sample_indices}, f)
print(f"  ✓ Saved: {sample_path}")

# ============================================================
# STEP 2: Word Alignment with SimAlign
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: WORD ALIGNMENT (SimAlign)")
print("=" * 60)
print(f"  Aligning {len(sample_mz):,} sentence pairs...")
print(f"  Estimated time: ~2.5 hours on RTX 3050")

from simalign import SentenceAligner
aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="i")
# Using only "i" (itermax) to save time — skip mwmf and inter
print(f"  ✓ SimAlign initialized (itermax only)")

start_time = time.time()
alignments = []
errors = []

# Process and save in chunks to avoid losing progress
CHUNK_SIZE = 10000
chunk_alignments_dir = os.path.join(OUTPUT_DIR, "align_chunks")
os.makedirs(chunk_alignments_dir, exist_ok=True)

for i in range(len(sample_mz)):
    try:
        en_tokens = sample_en[i].split()
        mz_tokens = sample_mz[i].split()
        
        if len(en_tokens) == 0 or len(mz_tokens) == 0:
            alignments.append([])
            continue
        
        result = aligner.get_word_aligns(en_tokens, mz_tokens)
        alignments.append(result.get("itermax", []))
        
    except Exception as e:
        alignments.append([])
        errors.append((i, str(e)))
    
    # Progress and checkpoint
    if (i + 1) % 2000 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(sample_mz) - (i + 1)) / rate
        print(f"    Aligned {i+1:>8,}/{len(sample_mz):,} "
              f"({100*(i+1)/len(sample_mz):.1f}%) "
              f"[{elapsed:.0f}s, ~{remaining:.0f}s remaining]")
    
    # Save checkpoint every CHUNK_SIZE
    if (i + 1) % CHUNK_SIZE == 0:
        chunk_num = (i + 1) // CHUNK_SIZE
        chunk_path = os.path.join(chunk_alignments_dir, f"chunk_{chunk_num:03d}.pkl")
        chunk_data = alignments[-CHUNK_SIZE:]
        with open(chunk_path, "wb") as f:
            pickle.dump(chunk_data, f)
        print(f"    💾 Checkpoint saved: chunk_{chunk_num:03d}.pkl")

elapsed = time.time() - start_time
print(f"\n  ✓ Alignment complete in {elapsed:.0f}s ({len(sample_mz)/elapsed:.1f} pairs/sec)")
if errors:
    print(f"  ⚠️  {len(errors)} errors")

# Save full alignments
align_path = os.path.join(OUTPUT_DIR, "large_sample_alignments.pkl")
with open(align_path, "wb") as f:
    pickle.dump(alignments, f)
print(f"  ✓ Saved: {align_path}")

# ============================================================
# STEP 3: Tag Projection + Lexicon + Corrections
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: TAG PROJECTION + LEXICON + CORRECTIONS")
print("=" * 60)

def normalize_quotes(text):
    for old, new in {'\u2018':"'", '\u2019':"'", '\u201C':'"', '\u201D':'"'}.items():
        text = text.replace(old, new)
    return text

def is_punctuation(token):
    return bool(re.match(r'^[^\w\s]+$', token)) or all(
        c in '.,!?;:\'"()-–—…/\\&*#@%+=<>~^|`[]{}' for c in token
    )

PART_WITH_PUNCT = {"a.", "a,", "ni."}

projected_large = []
total_tokens = 0
tagged_tokens = 0
source_stats = Counter()

for sent_idx in range(len(sample_mz)):
    mz_tokens = sample_mz[sent_idx].split()
    en_tagged = tagged_english[sent_idx]
    align_pairs = alignments[sent_idx]
    
    mz_sent = []
    for tok_idx, token in enumerate(mz_tokens):
        mz_sent.append({
            "token": token,
            "projected_tags": [],
            "lexicon_tags": [],
            "final_tag": None,
            "confidence": 0.0,
            "source": "untagged",
        })
    
    # Project EN tags via alignment
    for en_idx, mz_idx in align_pairs:
        if en_idx < len(en_tagged) and mz_idx < len(mz_sent):
            mz_sent[mz_idx]["projected_tags"].append(en_tagged[en_idx]["pos"])
    
    # Lexicon lookup
    for tok in mz_sent:
        clean = normalize_quotes(tok["token"].lower()).rstrip(".,!?;:'\"()-")
        clean_both = clean.lstrip("'\"")
        for candidate in [clean, clean_both]:
            if candidate in word_lexicon:
                tok["lexicon_tags"] = list(word_lexicon[candidate].keys())
                break
    
    # Resolve tags (same logic as small corpus)
    for tok in mz_sent:
        total_tokens += 1
        token = tok["token"]
        projected = tok["projected_tags"]
        lexicon = tok["lexicon_tags"]
        clean = normalize_quotes(token.lower()).rstrip(".,!?;:'\"()-")
        clean_both = clean.lstrip("'\"")
        token_lower = normalize_quotes(token.lower())
        
        # Punctuation
        if is_punctuation(token.strip()):
            tok["final_tag"] = "PUNCT"
            tok["confidence"] = 1.0
            tok["source"] = "punct"
            source_stats["punct"] += 1
            tagged_tokens += 1
            continue
        
        # PART with punct
        if token_lower in PART_WITH_PUNCT:
            tok["final_tag"] = "PART"
            tok["confidence"] = 0.90
            tok["source"] = "part_punct"
            source_stats["part_punct"] += 1
            tagged_tokens += 1
            continue
        
        # Numbers
        if re.match(r'^[\d,]+\.?\d*$', clean):
            tok["final_tag"] = "NUM"
            tok["confidence"] = 0.95
            tok["source"] = "number"
            source_stats["number"] += 1
            tagged_tokens += 1
            continue
        
        projected_counter = Counter(projected)
        
        if projected and lexicon:
            projected_best = projected_counter.most_common(1)[0][0]
            if projected_best in lexicon:
                tok["final_tag"] = projected_best
                tok["confidence"] = 0.95
                tok["source"] = "agreement"
                source_stats["agreement"] += 1
            else:
                # Prefer lexicon
                best_lt = None
                best_score = -1
                for lt in lexicon:
                    score = projected_counter.get(lt, 0)
                    if score > best_score:
                        best_score = score
                        best_lt = lt
                if best_lt and best_score > 0:
                    tok["final_tag"] = best_lt
                    tok["confidence"] = 0.85
                    tok["source"] = "lexicon_partial"
                    source_stats["lexicon_partial"] += 1
                else:
                    for c in [clean, clean_both]:
                        if c in word_lexicon:
                            tok["final_tag"] = word_lexicon[c].most_common(1)[0][0]
                            break
                    if not tok["final_tag"]:
                        tok["final_tag"] = lexicon[0]
                    tok["confidence"] = 0.75
                    tok["source"] = "lexicon_override"
                    source_stats["lexicon_override"] += 1
            tagged_tokens += 1
        
        elif projected and not lexicon:
            tok["final_tag"] = projected_counter.most_common(1)[0][0]
            tok["confidence"] = 0.60
            tok["source"] = "projection_only"
            source_stats["projection_only"] += 1
            tagged_tokens += 1
        
        elif not projected and lexicon:
            for c in [clean, clean_both]:
                if c in word_lexicon:
                    tok["final_tag"] = word_lexicon[c].most_common(1)[0][0]
                    break
            if not tok["final_tag"]:
                tok["final_tag"] = lexicon[0]
            tok["confidence"] = 0.70
            tok["source"] = "lexicon_only"
            source_stats["lexicon_only"] += 1
            tagged_tokens += 1
        
        else:
            # Suffix heuristics
            tag_found = False
            if clean_both.endswith("na") and len(clean_both) > 3:
                tok["final_tag"] = "NOUN"; tok["confidence"] = 0.55; tok["source"] = "suffix"; tag_found = True
            elif clean_both.endswith("tu") and len(clean_both) > 3:
                tok["final_tag"] = "NOUN"; tok["confidence"] = 0.55; tok["source"] = "suffix"; tag_found = True
            elif clean_both.endswith("te") and len(clean_both) > 4:
                tok["final_tag"] = "NOUN"; tok["confidence"] = 0.50; tok["source"] = "suffix"; tag_found = True
            elif clean_both.endswith("ah") and len(clean_both) > 3:
                tok["final_tag"] = "ADV"; tok["confidence"] = 0.50; tok["source"] = "suffix"; tag_found = True
            elif clean_both.endswith("in") and len(clean_both) > 3:
                tok["final_tag"] = "ADV"; tok["confidence"] = 0.50; tok["source"] = "suffix"; tag_found = True
            elif token[0].isupper() and len(clean_both) > 2:
                tok["final_tag"] = "PROPN"; tok["confidence"] = 0.50; tok["source"] = "capitalized"; tag_found = True
            
            if tag_found:
                source_stats[tok["source"]] += 1
                tagged_tokens += 1
            else:
                tok["final_tag"] = "X"
                tok["confidence"] = 0.20
                tok["source"] = "untagged"
                source_stats["untagged"] += 1
    
    # Apply correction rules
    for tok in mz_sent:
        clean = normalize_quotes(tok["token"].lower()).rstrip(".,!?;:'\"()-")
        clean_both = clean.lstrip("'\"")
        for candidate in [clean, clean_both]:
            if candidate in correction_rules:
                rule = correction_rules[candidate]
                if tok["final_tag"] in rule["wrong_tags"]:
                    tok["final_tag"] = rule["correct_tag"]
                    tok["confidence"] = max(tok["confidence"], 0.85)
                    tok["source"] = "corrected"
                break
    
    projected_large.append(mz_sent)
    
    if (sent_idx + 1) % 20000 == 0:
        print(f"    Projected {sent_idx+1:>8,}/{len(sample_mz):,}")

print(f"\n  Total tokens: {total_tokens:,}")
print(f"  Tagged tokens: {tagged_tokens:,} ({100*tagged_tokens/total_tokens:.1f}%)")

print(f"\n  Source distribution:")
for source, count in source_stats.most_common():
    print(f"    {source:<20} {count:>10,} ({100*count/total_tokens:.1f}%)")

# ============================================================
# STEP 4: Quality filtering and export
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: QUALITY FILTERING & EXPORT")
print("=" * 60)

# Filter: all tokens tagged, min confidence >= 0.60
hq_sents = []
for sent in projected_large:
    all_tagged = all(tok["final_tag"] and tok["final_tag"] != "X" for tok in sent)
    min_conf = min(tok["confidence"] for tok in sent) if sent else 0
    if all_tagged and min_conf >= 0.60:
        hq_sents.append([(tok["token"], tok["final_tag"]) for tok in sent])

print(f"  High-quality sentences (no X, min_conf>=0.60): {len(hq_sents):,}")
hq_tokens = sum(len(s) for s in hq_sents)
print(f"  High-quality tokens: {hq_tokens:,}")

# Tag distribution in HQ set
hq_tag_dist = Counter(tag for sent in hq_sents for _, tag in sent)
print(f"\n  HQ tag distribution:")
for tag, count in hq_tag_dist.most_common():
    print(f"    {tag:<10} {count:>10,} ({100*count/hq_tokens:.1f}%)")

# Export HQ large corpus
hq_path = os.path.join(OUTPUT_DIR, "large_hq_aligned.conll")
with open(hq_path, "w", encoding="utf-8") as f:
    for sent in hq_sents:
        for token, tag in sent:
            f.write(f"{token}\t{tag}\n")
        f.write("\n")
file_size = os.path.getsize(hq_path) / (1024 * 1024)
print(f"\n  ✓ {hq_path} ({file_size:.2f} MB, {len(hq_sents):,} sentences)")

# ============================================================
# STEP 5: Create combined training set
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: COMBINED TRAINING SET")
print("=" * 60)

# Load small corpus splits
def load_conll(filepath):
    sentences = []
    current = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split("\t")
                if len(parts) >= 2:
                    current.append((parts[0], parts[1]))
    if current:
        sentences.append(current)
    return sentences

small_train = load_conll(os.path.join(OUTPUT_DIR, "train.conll"))
small_dev = load_conll(os.path.join(OUTPUT_DIR, "dev.conll"))
small_test = load_conll(os.path.join(OUTPUT_DIR, "test.conll"))

print(f"  Small train: {len(small_train):,} sentences")
print(f"  Large HQ:    {len(hq_sents):,} sentences")

combined = small_train + hq_sents
random.seed(42)
random.shuffle(combined)  # Shuffle to mix small and large

combined_tokens = sum(len(s) for s in combined)
print(f"  Combined:    {len(combined):,} sentences ({combined_tokens:,} tokens)")

# Export combined
combined_path = os.path.join(OUTPUT_DIR, "train_combined.conll")
with open(combined_path, "w", encoding="utf-8") as f:
    for sent in combined:
        for token, tag in sent:
            f.write(f"{token}\t{tag}\n")
        f.write("\n")
file_size = os.path.getsize(combined_path) / (1024 * 1024)
print(f"  ✓ {combined_path} ({file_size:.2f} MB)")

# Combined tag distribution
combined_tags = Counter(tag for sent in combined for _, tag in sent)
print(f"\n  Combined tag distribution:")
for tag, count in combined_tags.most_common():
    print(f"    {tag:<10} {count:>10,} ({100*count/combined_tokens:.1f}%)")

print(f"\n  Dev (unchanged):  {len(small_dev):,} sentences")
print(f"  Test (unchanged): {len(small_test):,} sentences")

# Save projected large corpus
proj_path = os.path.join(OUTPUT_DIR, "large_sample_projected.pkl")
with open(proj_path, "wb") as f:
    pickle.dump(projected_large, f)
print(f"  ✓ {proj_path}")

print("\n" + "=" * 60)
print("Cell 7b Complete.")
print("=" * 60)
print(f"\nFinal training data:")
print(f"  train_combined.conll: {len(combined):,} sentences ({combined_tokens:,} tokens)")
print(f"  dev.conll:            {len(small_dev):,} sentences")
print(f"  test.conll:           {len(small_test):,} sentences")
print(f"\nNext: Cell 8 — Train BiLSTM-CRF PoS tagger")

Loading resources...
  Filtered corpus: 1,179,901 sentence pairs
  Word lexicon: 44,479
  Phrase lexicon: 22,924
  Correction rules: 296

  Sampled 200,000 sentence pairs for full pipeline

STEP 1: ENGLISH PoS TAGGING (spaCy)
  spaCy loaded: en_core_web_sm
  Tagging 200,000 English sentences...
    Tagged   20,000/200,000 (10.0%) [48s, ~430s remaining]
    Tagged   40,000/200,000 (20.0%) [92s, ~369s remaining]
    Tagged   60,000/200,000 (30.0%) [136s, ~317s remaining]
    Tagged   80,000/200,000 (40.0%) [180s, ~270s remaining]
    Tagged  100,000/200,000 (50.0%) [224s, ~224s remaining]
    Tagged  120,000/200,000 (60.0%) [268s, ~179s remaining]
    Tagged  140,000/200,000 (70.0%) [314s, ~135s remaining]
    Tagged  160,000/200,000 (80.0%) [361s, ~90s remaining]
    Tagged  180,000/200,000 (90.0%) [404s, ~45s remaining]
    Tagged  200,000/200,000 (100.0%) [449s, ~0s remaining]

  ✓ English tagging complete in 449s (445 sent/sec)
  ✓ Saved: data2\large_sample_en_tagged.pkl
  ✓ Saved: d

2026-02-17 20:42:45,460 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


  ✓ SimAlign initialized (itermax only)
    Aligned    2,000/200,000 (1.0%) [95s, ~9426s remaining]
    Aligned    4,000/200,000 (2.0%) [192s, ~9385s remaining]
    Aligned    6,000/200,000 (3.0%) [287s, ~9290s remaining]
    Aligned    8,000/200,000 (4.0%) [388s, ~9304s remaining]
    Aligned   10,000/200,000 (5.0%) [488s, ~9268s remaining]
    💾 Checkpoint saved: chunk_001.pkl
    Aligned   12,000/200,000 (6.0%) [585s, ~9162s remaining]
    Aligned   14,000/200,000 (7.0%) [683s, ~9070s remaining]
    Aligned   16,000/200,000 (8.0%) [780s, ~8970s remaining]
    Aligned   18,000/200,000 (9.0%) [877s, ~8867s remaining]
    Aligned   20,000/200,000 (10.0%) [974s, ~8764s remaining]
    💾 Checkpoint saved: chunk_002.pkl
    Aligned   22,000/200,000 (11.0%) [1071s, ~8666s remaining]
    Aligned   24,000/200,000 (12.0%) [1168s, ~8565s remaining]
    Aligned   26,000/200,000 (13.0%) [1265s, ~8465s remaining]
    Aligned   28,000/200,000 (14.0%) [1363s, ~8373s remaining]
    Aligned   30,000/2